---
## 📦 1. Instalación y Configuración

In [1]:
# Instalar librerías (solo necesario en Colab o primera ejecución local)
# En local, puedes instalar una vez con: pip install -r requirements.txt
try:
    import xgboost, sklearn, plotly, fredapi, yfinance, nbformat, jinja2, lightgbm, IPython, optuna
    print("Librerias ya instaladas")
except ImportError:
    %pip install -q openpyxl pandas numpy matplotlib seaborn scikit-learn statsmodels
    %pip install -q xgboost lightgbm plotly fredapi yfinance nbformat jinja2 ipython optuna
    print("Librerias instaladas - reinicia el kernel si es necesario")

C:\Users\dguaigua\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Librerias instaladas - reinicia el kernel si es necesario



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuración
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 10

print("✅ Librerías importadas")

✅ Librerías importadas


---
## 📁 2. Carga de Datos

In [3]:
# Detectar entorno: Google Colab o local
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta = '/content/drive/MyDrive/PRECIO SPOT BANANO/'
    print("Ejecutando en Google Colab")
except ModuleNotFoundError:
    # Ejecución local: colocar los archivos Excel en ../data/
    ruta = os.path.normpath(os.path.join(os.getcwd(), '..', 'data')) + os.sep
    print(f"Ejecutando localmente. Ruta de datos: {ruta}")
    if not os.path.exists(ruta):
        os.makedirs(ruta, exist_ok=True)
        print(f"AVISO: Carpeta '{ruta}' creada. Copiar los archivos Excel de Google Drive aqui:"
              "\n  - Enfunde_Colombia_Historico.xlsx"
              "\n  - Enfunde_Costa_Rica_Historico.xlsx"
              "\n  - Enfunde_Ecuador_Historico.xlsx"
              "\n  - Precios_Spot_Banano.xlsx"
              "\n  - Clima_Ecuador_Historico.xlsx"
              "\n  - Exportaciones_Ecuador.xlsx")

Ejecutando localmente. Ruta de datos: c:\Users\dguaigua\Documents\PROYECTOS\PROYECTO PRECIOS SPOT\data\


In [4]:
# Cargar todos los datasets
df_colombia = pd.read_excel(ruta + 'Enfunde_Colombia_Historico.xlsx')
df_costa_rica = pd.read_excel(ruta + 'Enfunde_Costa_Rica_Historico.xlsx')
df_ecuador = pd.read_excel(ruta + 'Enfunde_Ecuador_Historico.xlsx')
df_precios = pd.read_excel(ruta + 'Precios_Spot_Banano.xlsx')
df_clima = pd.read_excel(ruta + 'Datos_Clima_Historico.xlsx')
df_exportacion = pd.read_excel(ruta + 'Exportacion_semanal_Ecuador.xlsx')

# Cargar datos externos desde FRED y Yahoo Finance
from fredapi import Fred
import yfinance as yf
from datetime import datetime

# Obtener FRED API key segun el entorno
def get_fred_api_key():
    # 1. Intentar Colab Secrets (panel izquierdo > llave > agregar FRED_API_KEY)
    try:
        from google.colab import userdata
        return userdata.get('FRED_API_KEY')
    except (ModuleNotFoundError, Exception):
        pass
    # 2. Variable de entorno (local)
    key = os.environ.get('FRED_API_KEY')
    if key:
        return key
    raise ValueError("FRED_API_KEY no encontrada. Configurar como variable de entorno o en Colab Secrets.")

print("Descargando datos externos (FRED y Yahoo)...")
try:
    start_date = '2020-01-01'
    end_date = datetime.now()

    fred = Fred(api_key=get_fred_api_key())

    fred_series = {
        'DDFUELUSGULF': 'Diesel_Gulf',
        'DCOILWTICO': 'Petroleo_WTI',
        'DCOILBRENTEU': 'Petroleo_Brent',
        'PCU325311325311A': 'Fertilizante_Nitrogenado',
        'PCU483111483111': 'PPI_Fletes_Maritimos'
    }

    dfs_fred = []
    for ticker, col_name in fred_series.items():
        serie = fred.get_series(ticker, observation_start=start_date, observation_end=end_date)
        df_temp = serie.to_frame(name=col_name)
        dfs_fred.append(df_temp)

    df_commodities = pd.concat(dfs_fred, axis=1)

    # Resamplear a frecuencia semanal (W = termina en Domingo)
    df_commodities = df_commodities.resample('W').mean()
    df_commodities.reset_index(inplace=True)
    df_commodities.rename(columns={'index': 'Fecha'}, inplace=True)
    print("Datos de FRED descargados")

    # 2. Yahoo Finance: BDRY (Proxy Fletes Granel)
    df_shipping = yf.download('BDRY', start=start_date, progress=False)
    # Manejo de columna de cierre
    if 'Adj Close' in df_shipping.columns:
        serie_fletes = df_shipping['Adj Close']
    elif 'Close' in df_shipping.columns:
        serie_fletes = df_shipping['Close']
    else:
        serie_fletes = None

    if serie_fletes is not None:
        df_fletes = serie_fletes.resample('W').ffill().reset_index()
        df_fletes.columns = ['Fecha', 'Indice_Fletes_BDRY']
        df_fletes['Fecha'] = df_fletes['Fecha'].dt.tz_localize(None)
        print("Datos BDRY descargados")
    else:
        df_fletes = pd.DataFrame(columns=['Fecha', 'Indice_Fletes_BDRY'])

except Exception as e:
    print(f"Error descargando datos externos: {e}")
    # Dataframes vacios de respaldo
    df_commodities = pd.DataFrame(columns=['Fecha', 'Diesel_Gulf', 'Petroleo_WTI', 'Petroleo_Brent', 'Fertilizante_Nitrogenado', 'PPI_Fletes_Maritimos'])
    df_fletes = pd.DataFrame(columns=['Fecha', 'Indice_Fletes_BDRY'])

print("Carga de datos finalizada")
print(f"\nDimensiones:")
print(f"Ecuador: {df_ecuador.shape} (PRINCIPAL)")
print(f"Colombia: {df_colombia.shape}")
print(f"Costa Rica: {df_costa_rica.shape}")
print(f"Precios (OBJETIVO): {df_precios.shape}")
print(f"Clima: {df_clima.shape}")
print(f"Exportacion: {df_exportacion.shape}")
print(f"Commodities (FRED): {df_commodities.shape}")
print(f"Fletes (BDRY): {df_fletes.shape}")

Descargando datos externos (FRED y Yahoo)...
Datos de FRED descargados
Datos BDRY descargados
Carga de datos finalizada

Dimensiones:
Ecuador: (218, 6) (PRINCIPAL)
Colombia: (217, 3)
Costa Rica: (217, 3)
Precios (OBJETIVO): (270, 3)
Clima: (271, 6)
Exportacion: (214, 3)
Commodities (FRED): (325, 6)
Fletes (BDRY): (325, 2)


---
## 🔧 3. Preparación de Datos

In [5]:
# Función para crear fechas usando estándar ISO 8601 (Alineado a Domingo)
def crear_fecha(row):
    # Año, Semana, Día 7 (Domingo)
    # Esto asegura consistencia con el resample('W') de pandas/FRED que termina en Domingo
    return datetime.fromisocalendar(int(row['Año']), int(row['Semana']), 7)

# Asegurar que df_exportacion tenga las columnas 'Año' y 'Semana' en mayúsculas
if 'año' in df_exportacion.columns:
    df_exportacion.rename(columns={'año': 'Año'}, inplace=True)
if 'semana' in df_exportacion.columns:
    df_exportacion.rename(columns={'semana': 'Semana'}, inplace=True)

# Aplicar a todos los datasets que requieren construcción de fecha
for df in [df_colombia, df_costa_rica, df_ecuador, df_precios, df_clima, df_exportacion]:
    df['Fecha'] = df.apply(crear_fecha, axis=1)
    df.sort_values('Fecha', inplace=True)
    df.reset_index(drop=True, inplace=True)

# Preparar df_commodities (ya tiene fecha desde FRED, solo asegurar orden)
if 'df_commodities' in locals() and not df_commodities.empty:
    df_commodities.sort_values('Fecha', inplace=True)
    df_commodities.reset_index(drop=True, inplace=True)

# Limpiar precios (convertir de string a float)
df_precios['Precio'] = df_precios['Precio'].astype(str).str.replace(',', '.').astype(float)

print("✅ Fechas creadas y alineadas al estándar ISO (Domingo)")
print(f"\nRango de fechas Precios: {df_precios['Fecha'].min()} a {df_precios['Fecha'].max()}")
print(f"Rango de fechas Ecuador: {df_ecuador['Fecha'].min()} a {df_ecuador['Fecha'].max()}")
if 'df_commodities' in locals() and not df_commodities.empty:
    print(f"Rango de fechas Commodities (FRED): {df_commodities['Fecha'].min()} a {df_commodities['Fecha'].max()}")

✅ Fechas creadas y alineadas al estándar ISO (Domingo)

Rango de fechas Precios: 2021-01-10 00:00:00 a 2026-03-15 00:00:00
Rango de fechas Ecuador: 2022-01-09 00:00:00 a 2026-03-08 00:00:00
Rango de fechas Commodities (FRED): 2020-01-05 00:00:00 a 2026-03-22 00:00:00


In [6]:
df_commodities.to_excel('df_commodities.xlsx', index=False)

### 3.1 Crear Dataset Consolidado

In [7]:
# Asegurar renombrado de columnas (por si se saltó la celda anterior)
if 'Enfunde' in df_colombia.columns:
    df_colombia = df_colombia.rename(columns={'Enfunde': 'Enfunde_Colombia'})
if 'Enfunde' in df_costa_rica.columns:
    df_costa_rica = df_costa_rica.rename(columns={'Enfunde': 'Enfunde_CostaRica'})

# Renombrar Ecuador (safe rename)
df_ecuador = df_ecuador.rename(columns={
    'Ecuador': 'Enfunde_Ecuador',
    'Los Rios': 'Enfunde_LosRios',
    'Guayas': 'Enfunde_Guayas',
    'El Oro': 'Enfunde_ElOro'
})

if 'Cajas' in df_exportacion.columns:
    df_exportacion = df_exportacion.rename(columns={'Cajas': 'Exportaciones_Ecuador'})

# Unir todos los datos por fecha
# Empezamos con Precios (variable objetivo)
df_master = df_precios[['Fecha', 'Precio']].copy()

# Agregar enfunde Ecuador
df_master = df_master.merge(
    df_ecuador[['Fecha', 'Enfunde_Ecuador', 'Enfunde_LosRios', 'Enfunde_Guayas', 'Enfunde_ElOro']],
    on='Fecha', how='left'
)

# Agregar enfunde Colombia
df_master = df_master.merge(
    df_colombia[['Fecha', 'Enfunde_Colombia']],
    on='Fecha', how='left'
)

# Agregar enfunde Costa Rica
df_master = df_master.merge(
    df_costa_rica[['Fecha', 'Enfunde_CostaRica']],
    on='Fecha', how='left'
)

# Agregar datos climáticos
df_master = df_master.merge(
    df_clima[['Fecha', 'Temperatura', 'Precipitaciones', 'Radiación Solar', 'Humedad']],
    on='Fecha', how='left'
)

# Agregar datos de exportación
df_master = df_master.merge(
    df_exportacion[['Fecha', 'Exportaciones_Ecuador']],
    on='Fecha', how='left'
)

# Agregar commodities (FRED) - Actualizado
if 'df_commodities' in locals() and not df_commodities.empty:
    cols_fred = ['Fecha', 'Diesel_Gulf', 'Petroleo_WTI', 'Petroleo_Brent', 'Fertilizante_Nitrogenado', 'PPI_Fletes_Maritimos']
    cols_existentes = [c for c in cols_fred if c in df_commodities.columns]
    df_master = df_master.merge(
        df_commodities[cols_existentes],
        on='Fecha', how='left'
    )

# Agregar Fletes BDRY - NUEVO
if 'df_fletes' in locals() and not df_fletes.empty:
    df_master = df_master.merge(
        df_fletes[['Fecha', 'Indice_Fletes_BDRY']],
        on='Fecha', how='left'
    )

print(f"✅ Dataset consolidado creado: {df_master.shape}")

# Verificación
if 'Fertilizante_Nitrogenado' in df_master.columns:
    print(f"✅ Variable 'Fertilizante_Nitrogenado' integrada.")

print(f"\nPrimeras filas:")
print(df_master.head(5))

✅ Dataset consolidado creado: (270, 19)
✅ Variable 'Fertilizante_Nitrogenado' integrada.

Primeras filas:
       Fecha  Precio  Enfunde_Ecuador  Enfunde_LosRios  Enfunde_Guayas  \
0 2021-01-10    7.04              NaN              NaN             NaN   
1 2021-01-17    7.24              NaN              NaN             NaN   
2 2021-01-24    7.61              NaN              NaN             NaN   
3 2021-01-31    9.00              NaN              NaN             NaN   
4 2021-02-07   10.21              NaN              NaN             NaN   

   Enfunde_ElOro  Enfunde_Colombia  Enfunde_CostaRica  Temperatura  \
0            NaN               NaN                NaN        25.79   
1            NaN               NaN                NaN        26.25   
2            NaN               NaN                NaN        25.97   
3            NaN               NaN                NaN        25.91   
4            NaN               NaN                NaN        26.44   

   Precipitaciones  Radiació

In [8]:
# Verificar valores faltantes
print("\nValores faltantes por columna:")
print(df_master.isnull().sum())
print(f"\nPorcentaje de completitud:")
print((1 - df_master.isnull().sum() / len(df_master)) * 100)


Valores faltantes por columna:
Fecha                         0
Precio                        0
Enfunde_Ecuador              53
Enfunde_LosRios              53
Enfunde_Guayas               53
Enfunde_ElOro                53
Enfunde_Colombia             54
Enfunde_CostaRica            54
Temperatura                   0
Precipitaciones               0
Radiación Solar               0
Humedad                       0
Exportaciones_Ecuador        57
Diesel_Gulf                   0
Petroleo_WTI                  0
Petroleo_Brent                0
Fertilizante_Nitrogenado    209
PPI_Fletes_Maritimos        209
Indice_Fletes_BDRY            0
dtype: int64

Porcentaje de completitud:
Fecha                       100.000000
Precio                      100.000000
Enfunde_Ecuador              80.370370
Enfunde_LosRios              80.370370
Enfunde_Guayas               80.370370
Enfunde_ElOro                80.370370
Enfunde_Colombia             80.000000
Enfunde_CostaRica            80.000000
Tempera

# 🤖 **4. XGBOOST**

## 4.1 Imputar Valores Faltantes



In [9]:
print("\n" + "="*80)
print("IMPUTACION DE VALORES FALTANTES (INTERPOLACION + FFILL/BFILL)")
print("="*80)

# Reportar % de valores faltantes antes de imputar
print("\nPorcentaje de valores faltantes ANTES de imputar:")
missing_pct = (df_master.isnull().sum() / len(df_master) * 100).round(1)
for col, pct in missing_pct[missing_pct > 0].items():
    print(f"  {col}: {pct}% missing")

# Eliminar columnas con >50% de valores faltantes (datos mayormente imputados no aportan senal)
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
if cols_to_drop:
    df_master.drop(columns=cols_to_drop, inplace=True)
    print(f"\nColumnas eliminadas por >50% missing: {cols_to_drop}")

# Aplicar interpolacion lineal para variables numericas continuas
# (mejor que ffill para series temporales - preserva tendencia local)
numeric_cols = df_master.select_dtypes(include='number').columns
df_master[numeric_cols] = df_master[numeric_cols].interpolate(method='linear', limit_direction='both')
print("Valores faltantes interpolados linealmente (variables numericas).")

# Aplicar ffill + bfill solo para valores remanentes en bordes
df_master = df_master.ffill()
df_master = df_master.bfill()
print("Valores remanentes en bordes rellenados con ffill/bfill.")

print("\nVerificacion de valores faltantes despues de la imputacion:")
missing_values_after_imputation = df_master.isnull().sum()
print(missing_values_after_imputation[missing_values_after_imputation > 0])

if missing_values_after_imputation.sum() == 0:
    print("\nNo quedan valores faltantes en el DataFrame `df_master`.")
else:
    print("\nTodavia quedan valores faltantes en algunas columnas.")

print("\n" + "="*80)


IMPUTACION DE VALORES FALTANTES (INTERPOLACION + FFILL/BFILL)

Porcentaje de valores faltantes ANTES de imputar:
  Enfunde_Ecuador: 19.6% missing
  Enfunde_LosRios: 19.6% missing
  Enfunde_Guayas: 19.6% missing
  Enfunde_ElOro: 19.6% missing
  Enfunde_Colombia: 20.0% missing
  Enfunde_CostaRica: 20.0% missing
  Exportaciones_Ecuador: 21.1% missing
  Fertilizante_Nitrogenado: 77.4% missing
  PPI_Fletes_Maritimos: 77.4% missing

Columnas eliminadas por >50% missing: ['Fertilizante_Nitrogenado', 'PPI_Fletes_Maritimos']
Valores faltantes interpolados linealmente (variables numericas).
Valores remanentes en bordes rellenados con ffill/bfill.

Verificacion de valores faltantes despues de la imputacion:
Series([], dtype: int64)

No quedan valores faltantes en el DataFrame `df_master`.



## 4.2 Generar Características Temporales

Se crearon nuevas columnas a partir de la fecha para capturar patrones estacionales y cíclicos: 'Mes', 'Semana del Año', 'Día de la Semana', 'Día del Año'.


In [10]:
print("\n" + "="*80)
print("GENERACION DE CARACTERISTICAS TEMPORALES")
print("="*80)

# 1. Extraer el mes
df_master['Mes'] = df_master['Fecha'].dt.month
print("✅ Columna 'Mes' creada.")

# 2. Extraer la semana del año
df_master['Semana del Año'] = df_master['Fecha'].dt.isocalendar().week.astype(int)
print("✅ Columna 'Semana del Año' creada.")

# NOTA: 'Día de la Semana' y 'Día del Año' fueron eliminados porque son constantes
# (todos los registros son domingos, siempre valor 6 y el mismo día del año por semana)
print("⚠️ 'Día de la Semana' y 'Día del Año' NO se crean (son constantes en datos semanales dominicales).")

print("\nPrimeras filas de df_master con las nuevas caracteristicas temporales:")
print(df_master.head())
print("\n" + "="*80)


GENERACION DE CARACTERISTICAS TEMPORALES
✅ Columna 'Mes' creada.
✅ Columna 'Semana del Año' creada.
⚠️ 'Día de la Semana' y 'Día del Año' NO se crean (son constantes en datos semanales dominicales).

Primeras filas de df_master con las nuevas caracteristicas temporales:
       Fecha  Precio  Enfunde_Ecuador  Enfunde_LosRios  Enfunde_Guayas  \
0 2021-01-10    7.04        37.438897        41.044218       40.205766   
1 2021-01-17    7.24        37.438897        41.044218       40.205766   
2 2021-01-24    7.61        37.438897        41.044218       40.205766   
3 2021-01-31    9.00        37.438897        41.044218       40.205766   
4 2021-02-07   10.21        37.438897        41.044218       40.205766   

   Enfunde_ElOro  Enfunde_Colombia  Enfunde_CostaRica  Temperatura  \
0      29.226038         50.339866               54.0        25.79   
1      29.226038         50.339866               54.0        26.25   
2      29.226038         50.339866               54.0        25.97   
3  

In [11]:
print("\n" + "="*80)
print("🔬 GENERACION DE CARACTERISTICAS LAGGED - BASADO EN GRANGER & CCF")
print("="*80)

# ================================================================
# CONFIGURACION DE LAGS BASADA EN ANALISIS DE CAUSALIDAD
# ================================================================

# VARIABLES CON CAUSALIDAD DE GRANGER + CORRELACION CRUZADA
predictor_lags_config = {
    # ALTA PRIORIDAD - Causalidad fuerte en Granger
    'Enfunde_Ecuador': {
        'lags': [1, 2, 3, 4, 12],  # Granger: [1-4], CCF pico: 12
        'prioridad': 'ALTA',
        'granger': True
    },
    'Temperatura': {
        'lags': [2, 3, 4, 5, 8, 12],  # Granger: [2-8], CCF pico: 12 (reducido de 8 a 6 lags)
        'prioridad': 'ALTA',
        'granger': True
    },
    'Radiación Solar': {
        'lags': [2, 4, 7, 12],  # Granger: [2-7], CCF pico: 12 (reducido de 7 a 4 lags)
        'prioridad': 'ALTA',
        'granger': True
    },

    # PRIORIDAD MEDIA - Causalidad moderada
    'Exportaciones_Ecuador': {
        'lags': [1, 2, 12],  # Granger: [1-2], CCF pico: 12
        'prioridad': 'MEDIA',
        'granger': True
    },
    'Precipitaciones': {
        'lags': [2],  # Granger: [2], CCF pico: 2
        'prioridad': 'MEDIA',
        'granger': True
    },
    'Humedad': {
        'lags': [2, 12],  # Granger: [2], CCF pico: 12
        'prioridad': 'MEDIA',
        'granger': True
    },
    'Petroleo_Brent': {
        'lags': [1, 10],  # Granger: [1], CCF pico: 10
        'prioridad': 'MEDIA',
        'granger': True
    },

    # EXPERIMENTAL - Alta correlacion pero no paso Granger
    'Indice_Fletes_BDRY': {
        'lags': [4],  # CCF muy alto: -0.564 en lag 4
        'prioridad': 'EXPERIMENTAL',
        'granger': False,
        'nota': 'Correlacion cruzada muy fuerte'
    },
    'Petroleo_WTI': {
        'lags': [10],  # CCF: -0.553 en lag 10
        'prioridad': 'EXPERIMENTAL',
        'granger': False,
        'nota': 'Alta correlacion (puede ser colineal con Brent)'
    },

    # NOTA: Fertilizante_Nitrogenado fue ELIMINADO
    # Razon: ~78% de valores son imputados (solo 58 obs reales de 270)
    # Un feature mayormente imputado agrega ruido, no senal predictiva
}

# ================================================================
# CREAR CARACTERISTICAS LAGGED
# ================================================================

feature_names_created = []
stats = {'ALTA': 0, 'MEDIA': 0, 'EXPERIMENTAL': 0, 'BAJA': 0}

print("\n📊 CREANDO FEATURES LAGGED:\n")

for col, config in predictor_lags_config.items():
    if col in df_master.columns:
        prioridad = config['prioridad']
        granger_status = "✅ Granger" if config['granger'] else "⚠️ Solo CCF"

        print(f"\n{col} [{prioridad}] - {granger_status}")

        for lag in config['lags']:
            col_name = f'{col}_lag{lag}'
            df_master[col_name] = df_master[col].shift(lag)
            feature_names_created.append(col_name)
            stats[prioridad] += 1
            print(f"  -> {col_name}")

        if 'nota' in config:
            print(f"  📝 {config['nota']}")
    else:
        print(f"⚠️ Variable '{col}' no encontrada en df_master (posiblemente eliminada por alto % missing)")

# ================================================================
# CARACTERISTICAS LAGGED DE LA VARIABLE OBJETIVO (Precio)
# ================================================================

print("\n" + "-"*80)
print("🎯 LAGS DE LA VARIABLE OBJETIVO (Precio)")
print("-"*80)

# Lags autorregresivos para series temporales semanales
precio_lags = [1, 2, 4, 8, 12, 52]  # Eliminado lag24 (poco informativo)

for lag in precio_lags:
    col_name = f'Precio_lag{lag}'
    df_master[col_name] = df_master['Precio'].shift(lag)
    feature_names_created.append(col_name)

    if lag == 1:
        desc = "semana anterior"
    elif lag == 52:
        desc = "mismo periodo año anterior"
    elif lag == 12:
        desc = "~3 meses"
    else:
        desc = f"{lag} semanas"

    print(f"✅ {col_name:20s} ({desc})")

# ================================================================
# FEATURES DE MOMENTUM / ROLLING (NUEVOS)
# ================================================================

print("\n" + "-"*80)
print("📈 FEATURES DE MOMENTUM Y ROLLING (NUEVOS)")
print("-"*80)

# Rolling mean y std capturan tendencia y volatilidad
for window in [4, 8, 12]:
    col_mean = f'Precio_rolling_mean_{window}'
    col_std = f'Precio_rolling_std_{window}'
    df_master[col_mean] = df_master['Precio'].shift(1).rolling(window=window).mean()
    df_master[col_std] = df_master['Precio'].shift(1).rolling(window=window).std()
    feature_names_created.extend([col_mean, col_std])
    print(f"✅ {col_mean}, {col_std}")

# Rate of change (momentum)
df_master['Precio_roc_4'] = (df_master['Precio'].shift(1) - df_master['Precio'].shift(5)) / df_master['Precio'].shift(5)
feature_names_created.append('Precio_roc_4')
print("✅ Precio_roc_4 (rate of change 4 semanas)")

# ================================================================
# CARACTERISTICAS TEMPORALES ADICIONALES
# ================================================================

print("\n" + "-"*80)
print("📅 CARACTERISTICAS TEMPORALES")
print("-"*80)

if 'Fecha' in df_master.columns:
    df_master['Fecha'] = pd.to_datetime(df_master['Fecha'])

    # Features ciclicas para capturar estacionalidad
    df_master['semana_año'] = df_master['Fecha'].dt.isocalendar().week

    # Componentes trigonometricos para estacionalidad (mejor que categoricas)
    df_master['semana_sin'] = np.sin(2 * np.pi * df_master['semana_año'] / 52)
    df_master['semana_cos'] = np.cos(2 * np.pi * df_master['semana_año'] / 52)
    df_master['mes_sin'] = np.sin(2 * np.pi * df_master['Fecha'].dt.month / 12)
    df_master['mes_cos'] = np.cos(2 * np.pi * df_master['Fecha'].dt.month / 12)

    # NOTA: Eliminados 'mes', 'trimestre', 'dia_año' por redundancia con sin/cos
    # Los componentes trigonometricos ya capturan la estacionalidad sin crear
    # features categoricas que XGBoost trata de forma suboptima
    temporal_features = [
        'semana_año',
        'semana_sin', 'semana_cos', 'mes_sin', 'mes_cos'
    ]

    for feat in temporal_features:
        feature_names_created.append(feat)
        print(f"✅ {feat}")
else:
    print("⚠️ No se encontro columna 'Fecha'")
    temporal_features = []

# Tendencia lineal
df_master['tendencia'] = np.arange(len(df_master))
feature_names_created.append('tendencia')
print(f"✅ tendencia")

# ================================================================
# RESUMEN
# ================================================================

print("\n" + "="*80)
print("📈 RESUMEN DE FEATURES GENERADAS")
print("="*80)

print(f"\n🎯 Features de predictores:")
for k, v in stats.items():
    if v > 0:
        print(f"   - {k}: {v:3d} features")
print(f"   - Subtotal:            {sum(stats.values()):3d} features")

print(f"\n📊 Features de variable objetivo:")
print(f"   - Lags de Precio:      {len(precio_lags):3d} features")
print(f"   - Rolling/Momentum:      7 features")

print(f"\n📅 Features temporales:")
print(f"   - Temporales:          {len(temporal_features) + 1:3d} features")

print(f"\n✨ TOTAL FEATURES:        {len(feature_names_created):3d} features")

# Calcular filas validas despues de aplicar lags
filas_perdidas = df_master[feature_names_created].isna().any(axis=1).sum()
filas_validas = len(df_master) - filas_perdidas

print(f"\n📉 Filas totales:         {len(df_master):3d}")
print(f"   Filas con NaN (lags):  {filas_perdidas:3d}")
print(f"   Filas validas:         {filas_validas:3d}")
print(f"   Ratio obs/features:    {filas_validas/len(feature_names_created):.1f}")

# Mostrar primeras y ultimas filas
print("\n" + "-"*80)
print("🔍 PREVIEW DE DATOS (ultimas 5 filas):")
print("-"*80)
display_cols = ['Fecha', 'Precio'] + [col for col in feature_names_created[:8]]
print(df_master[display_cols].tail())

# Guardar lista de features para uso posterior
print("\n💾 Lista de features guardada en 'feature_names_ml'")
feature_names_ml = feature_names_created.copy()

print("\n" + "="*80 + "\n")


🔬 GENERACION DE CARACTERISTICAS LAGGED - BASADO EN GRANGER & CCF

📊 CREANDO FEATURES LAGGED:


Enfunde_Ecuador [ALTA] - ✅ Granger
  -> Enfunde_Ecuador_lag1
  -> Enfunde_Ecuador_lag2
  -> Enfunde_Ecuador_lag3
  -> Enfunde_Ecuador_lag4
  -> Enfunde_Ecuador_lag12

Temperatura [ALTA] - ✅ Granger
  -> Temperatura_lag2
  -> Temperatura_lag3
  -> Temperatura_lag4
  -> Temperatura_lag5
  -> Temperatura_lag8
  -> Temperatura_lag12

Radiación Solar [ALTA] - ✅ Granger
  -> Radiación Solar_lag2
  -> Radiación Solar_lag4
  -> Radiación Solar_lag7
  -> Radiación Solar_lag12

Exportaciones_Ecuador [MEDIA] - ✅ Granger
  -> Exportaciones_Ecuador_lag1
  -> Exportaciones_Ecuador_lag2
  -> Exportaciones_Ecuador_lag12

Precipitaciones [MEDIA] - ✅ Granger
  -> Precipitaciones_lag2

Humedad [MEDIA] - ✅ Granger
  -> Humedad_lag2
  -> Humedad_lag12

Petroleo_Brent [MEDIA] - ✅ Granger
  -> Petroleo_Brent_lag1
  -> Petroleo_Brent_lag10

Indice_Fletes_BDRY [EXPERIMENTAL] - ⚠️ Solo CCF
  -> Indice_Fletes_BDRY_lag

In [12]:
df_master.to_csv('df_master_final.csv', index=False)

## 4.3 Definir Función de Métricas de Rendimiento

In [13]:
print("\n" + "="*80)
print("DEFINIENDO FUNCIONES DE METRICAS DE EVALUACION")
print("="*80)

def evaluar_modelo(y_true, y_pred):
    """
    Calcula las metricas de evaluacion RMSE, MAE, MAPE y R2.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_true_no_zero = np.where(y_true == 0, 1e-10, y_true)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true_no_zero)) * 100
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else float('nan')

    return {
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R2': r2
    }

def directional_accuracy(y_true, y_pred):
    """
    Calcula el % de veces que el modelo predice correctamente
    la direccion del cambio (sube/baja) respecto al valor anterior.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    if len(y_true) < 2:
        return float('nan')

    actual_direction = np.diff(y_true) >= 0
    pred_direction = (y_pred[1:] - y_true[:-1]) >= 0
    return np.mean(actual_direction == pred_direction) * 100

print("✅ Funcion 'evaluar_modelo' definida (RMSE, MAE, MAPE, R2).")
print("✅ Funcion 'directional_accuracy' definida (% acierto direccional).")
print("\n" + "="*80)


DEFINIENDO FUNCIONES DE METRICAS DE EVALUACION
✅ Funcion 'evaluar_modelo' definida (RMSE, MAE, MAPE, R2).
✅ Funcion 'directional_accuracy' definida (% acierto direccional).



## 4.4 Implementación y evaluación del modelo

In [14]:
# Importar librerías adicionales necesarias
import xgboost as xgb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print("✅ XGBoost y Plotly importados correctamente")

✅ XGBoost y Plotly importados correctamente


In [15]:
print("\n" + "="*80)
print("⚙️  CONFIGURACIÓN DE PARÁMETROS XGBOOST")
print("="*80)

xgb_params = {
    'n_estimators': 200,          # ⬇️ Reducir de 300
    'max_depth': 5,               # ⬇️ Reducir de 10 (CRÍTICO)
    'learning_rate': 0.05,        # ✅ Mantener
    'min_child_weight': 10,       # ⬆️ Aumentar de 5
    'subsample': 0.7,             # ⬇️ Reducir de 0.8
    'colsample_bytree': 0.7,      # ⬇️ Reducir de 0.8
    'gamma': 0.5,                 # ⬆️ Aumentar de 0.1 (CRÍTICO)
    'reg_alpha': 0.5,             # ⬆️ Aumentar de 0.1
    'reg_lambda': 3.0,            # ⬆️ Aumentar de 1.0
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

print("Parámetros configurados:")
for key, value in xgb_params.items():
    print(f"  • {key}: {value}")
print("="*80)


⚙️  CONFIGURACIÓN DE PARÁMETROS XGBOOST
Parámetros configurados:
  • n_estimators: 200
  • max_depth: 5
  • learning_rate: 0.05
  • min_child_weight: 10
  • subsample: 0.7
  • colsample_bytree: 0.7
  • gamma: 0.5
  • reg_alpha: 0.5
  • reg_lambda: 3.0
  • random_state: 42
  • n_jobs: -1
  • verbosity: 0


In [16]:
def walk_forward_validation_xgb(df, features, target='Precio',
                                initial_train_size=150,
                                test_size=1,
                                params=None):
    """
    Implementa validación Walk-Forward para series temporales con XGBoost.

    Parámetros:
    -----------
    df : DataFrame
        Dataset completo con features y target
    features : list
        Lista de nombres de columnas features
    target : str
        Nombre de la columna target
    initial_train_size : int
        Tamaño inicial del conjunto de entrenamiento
    test_size : int
        Número de observaciones a predecir en cada paso
    params : dict
        Parámetros del modelo XGBoost

    Retorna:
    --------
    dict con resultados de la validación
    """

    if params is None:
        params = xgb_params

    # Preparar datos
    df_clean = df.dropna(subset=features + [target]).reset_index(drop=True)

    # Listas para almacenar resultados
    predictions = []
    actuals = []
    train_metrics = []
    test_metrics = []
    fold_info = []
    models = []

    n_samples = len(df_clean)
    n_folds = (n_samples - initial_train_size) // test_size

    print("="*80)
    print("🚀 INICIANDO WALK-FORWARD VALIDATION CON XGBOOST")
    print("="*80)
    print(f"Total de observaciones: {n_samples}")
    print(f"Tamaño inicial de entrenamiento: {initial_train_size}")
    print(f"Tamaño de test por fold: {test_size}")
    print(f"Número de folds: {n_folds}")
    print("="*80)

    for fold in range(n_folds):
        # Definir índices de train y test
        train_start = 0
        train_end = initial_train_size + (fold * test_size)
        test_start = train_end
        test_end = test_start + test_size

        # Validar que no excedamos el dataset
        if test_end > n_samples:
            break

        # Split train/test
        X_train = df_clean.loc[train_start:train_end-1, features]
        y_train = df_clean.loc[train_start:train_end-1, target]
        X_test = df_clean.loc[test_start:test_end-1, features]
        y_test = df_clean.loc[test_start:test_end-1, target]

        # Entrenar modelo XGBoost
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, verbose=False)

        # Predicciones
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Métricas
        train_eval = evaluar_modelo(y_train, y_train_pred)
        test_eval = evaluar_modelo(y_test, y_test_pred)

        # Almacenar resultados
        predictions.extend(y_test_pred)
        actuals.extend(y_test)
        train_metrics.append(train_eval)
        test_metrics.append(test_eval)
        models.append(model)

        fold_info.append({
            'fold': fold + 1,
            'train_size': len(X_train),
            'test_size': len(X_test),
            'train_start': train_start,
            'train_end': train_end,
            'test_start': test_start,
            'test_end': test_end
        })

        # Mostrar progreso
        if (fold + 1) % 10 == 0 or fold == 0 or fold == n_folds - 1:
            print(f"Fold {fold+1}/{n_folds} | Train: {train_start}-{train_end-1} | "
                  f"Test: {test_start}-{test_end-1} | "
                  f"Test RMSE: {test_eval['RMSE']:.4f} | "
                  f"Test MAPE: {test_eval['MAPE']:.2f}%")

    # Calcular métricas agregadas
    predictions_array = np.array(predictions)
    actuals_array = np.array(actuals)
    overall_metrics = evaluar_modelo(actuals_array, predictions_array)

    print("="*80)
    print("📊 MÉTRICAS GENERALES WALK-FORWARD VALIDATION (XGBOOST)")
    print("="*80)
    print(f"RMSE: {overall_metrics['RMSE']:.4f}")
    print(f"MAE:  {overall_metrics['MAE']:.4f}")
    print(f"MAPE: {overall_metrics['MAPE']:.2f}%")
    print(f"R²:   {overall_metrics['R2']:.4f}")
    print("="*80)

    # Métricas promedio de train y test
    avg_train_metrics = {
        'RMSE': np.mean([m['RMSE'] for m in train_metrics]),
        'MAE': np.mean([m['MAE'] for m in train_metrics]),
        'MAPE': np.mean([m['MAPE'] for m in train_metrics]),
        'R2': np.mean([m['R2'] for m in train_metrics])
    }

    avg_test_metrics = {
        'RMSE': np.mean([m['RMSE'] for m in test_metrics]),
        'MAE': np.mean([m['MAE'] for m in test_metrics]),
        'MAPE': np.mean([m['MAPE'] for m in test_metrics]),
        'R2': np.mean([m['R2'] for m in test_metrics])
    }

    print("\n📈 PROMEDIO MÉTRICAS DE TRAIN")
    print(f"RMSE: {avg_train_metrics['RMSE']:.4f}")
    print(f"MAE:  {avg_train_metrics['MAE']:.4f}")
    print(f"MAPE: {avg_train_metrics['MAPE']:.2f}%")
    print(f"R²:   {avg_train_metrics['R2']:.4f}")

    print("\n📉 PROMEDIO MÉTRICAS DE TEST")
    print(f"RMSE: {avg_test_metrics['RMSE']:.4f}")
    print(f"MAE:  {avg_test_metrics['MAE']:.4f}")
    print(f"MAPE: {avg_test_metrics['MAPE']:.2f}%")
    print(f"R²:   {avg_test_metrics['R2']:.4f}")
    print("="*80)

    return {
        'predictions': predictions_array,
        'actuals': actuals_array,
        'overall_metrics': overall_metrics,
        'avg_train_metrics': avg_train_metrics,
        'avg_test_metrics': avg_test_metrics,
        'train_metrics_by_fold': train_metrics,
        'test_metrics_by_fold': test_metrics,
        'fold_info': fold_info,
        'models': models,
        'last_model': models[-1] if models else None,
        'df_clean': df_clean
    }

print("✅ Función walk_forward_validation_xgb definida")

✅ Función walk_forward_validation_xgb definida


In [17]:
def predict_future_weeks_xgb(df, features, target='Precio', n_weeks=4, params=None):
    """
    Predice las próximas n_weeks usando XGBoost entrenado con todos los datos.

    Parámetros:
    -----------
    df : DataFrame
        Dataset completo con features y target
    features : list
        Lista de nombres de features
    target : str
        Nombre de la columna target
    n_weeks : int
        Número de semanas a predecir hacia adelante
    params : dict
        Parámetros del modelo XGBoost

    Retorna:
    --------
    DataFrame con predicciones futuras, modelo final, métricas históricas
    """

    if params is None:
        params = xgb_params

    print("\n" + "="*80)
    print(f"🔮 PREDICCIÓN DE LAS PRÓXIMAS {n_weeks} SEMANAS CON XGBOOST")
    print("="*80)

    # Preparar datos limpios
    df_clean = df.dropna(subset=features + [target]).reset_index(drop=True)

    # Entrenar modelo con todos los datos disponibles
    X_full = df_clean[features]
    y_full = df_clean[target]

    print(f"Entrenando modelo final con {len(X_full)} observaciones...")
    model_final = xgb.XGBRegressor(**params)
    model_final.fit(X_full, y_full, verbose=False)

    # Predicción en datos históricos para validación
    y_pred_historical = model_final.predict(X_full)
    hist_metrics = evaluar_modelo(y_full, y_pred_historical)

    print(f"✅ Modelo final entrenado")
    print(f"   RMSE: {hist_metrics['RMSE']:.4f}")
    print(f"   MAPE: {hist_metrics['MAPE']:.2f}%")
    print(f"   R²:   {hist_metrics['R2']:.4f}")

    # Obtener última fecha del df_clean
    if 'Fecha' in df_clean.columns:
        last_date_clean = pd.to_datetime(df_clean['Fecha'].iloc[-1])
    else:
        last_date_clean = datetime.now()

    print(f"\nÚltima fecha en datos históricos: {last_date_clean.strftime('%Y-%m-%d')}")

    # Crear estructura para predicciones futuras
    future_predictions = []
    future_dates = []

    # DataFrame temporal que se extenderá con las predicciones para calcular los lags correctamente
    df_temp_extended = df_clean.copy()

    for week in range(1, n_weeks + 1):
        # Calcular fecha para la semana futura actual
        future_date = last_date_clean + timedelta(weeks=week)

        # Crear una nueva fila vacía para la predicción, con todas las columnas de df_temp_extended
        new_pred_row = pd.DataFrame(index=[0], columns=df_temp_extended.columns)
        new_pred_row['Fecha'] = future_date
        new_pred_row['Precio'] = np.nan # El precio se llenará después de la predicción

        # --- Llenar características temporales ---
        # Asegúrate de que estas columnas existan en `new_pred_row` antes de asignarlas
        if 'Mes' in new_pred_row.columns: new_pred_row['Mes'] = future_date.month
        if 'Semana del Año' in new_pred_row.columns: new_pred_row['Semana del Año'] = future_date.isocalendar().week
        if 'Día de la Semana' in new_pred_row.columns: new_pred_row['Día de la Semana'] = future_date.dayofweek
        if 'Día del Año' in new_pred_row.columns: new_pred_row['Día del Año'] = future_date.timetuple().tm_yday
        if 'tendencia' in new_pred_row.columns: new_pred_row['tendencia'] = len(df_temp_extended) # Tendencia es un contador secuencial

        if 'semana_sin' in new_pred_row.columns: new_pred_row['semana_sin'] = np.sin(2 * np.pi * future_date.isocalendar().week / 52)
        if 'semana_cos' in new_pred_row.columns: new_pred_row['semana_cos'] = np.cos(2 * np.pi * future_date.isocalendar().week / 52)
        if 'mes_sin' in new_pred_row.columns: new_pred_row['mes_sin'] = np.sin(2 * np.pi * future_date.month / 12)
        if 'mes_cos' in new_pred_row.columns: new_pred_row['mes_cos'] = np.cos(2 * np.pi * future_date.month / 12)

        # --- Llenar características lagged (desplazadas correctamente) ---
        for feature_col in features:
            if feature_col in new_pred_row.columns and pd.isna(new_pred_row[feature_col].iloc[0]): # Solo si aún no se ha llenado (ej. temporales)
                if '_lag' in feature_col:
                    parts = feature_col.split('_lag')
                    original_feature_name = parts[0]
                    lag_val = int(parts[1])

                    # El valor lagged se toma de la columna 'original_feature_name'
                    # 'lag_val' semanas antes del final actual de df_temp_extended
                    if original_feature_name in df_temp_extended.columns and (len(df_temp_extended) - lag_val) >= 0:
                        new_pred_row[feature_col] = df_temp_extended[original_feature_name].iloc[len(df_temp_extended) - lag_val]
                    else:
                        # Si el lag excede los datos disponibles, usar NaN
                        new_pred_row[feature_col] = np.nan
                else:
                    # --- Llenar características no lagged ni temporales (e.g., Diesel_Gulf si no está lagged) ---
                    # Asumimos que estas se propagan, tomando el último valor conocido de df_temp_extended
                    if feature_col in df_temp_extended.columns:
                        new_pred_row[feature_col] = df_temp_extended[feature_col].iloc[-1]
                    else:
                        new_pred_row[feature_col] = np.nan # Debería estar en df_temp_extended si está en features

        # Extraer características para la predicción, rellenando cualquier NaN restante
        X_future = new_pred_row[features].copy()
        # Rellenar cualquier NaN que pudiera haberse creado por lags muy grandes o faltantes.
        # XGBoost no maneja NaNs directamente sin configuración específica.
        X_future = X_future.ffill(axis=1).bfill(axis=1)  # Rellenar horizontalmente
        X_future = X_future.fillna(0) # Si aún quedan NaNs (e.g., si ffill/bfill no pudieron), rellenar con 0

        # Hacer predicción
        pred = model_final.predict(X_future)[0]

        future_predictions.append(pred)
        future_dates.append(future_date)

        print(f"Semana {week} ({future_date.strftime('%Y-%m-%d')}): ${pred:.2f}")

        # Actualizar la columna 'Precio' de la nueva fila con la predicción
        new_pred_row['Precio'] = pred

        # Concatenar la nueva fila predicha a df_temp_extended para las siguientes iteraciones
        df_temp_extended = pd.concat([df_temp_extended, new_pred_row], ignore_index=True)

    # Crear DataFrame con predicciones
    df_future = pd.DataFrame({
        'Fecha': future_dates,
        'Precio_Predicho': future_predictions,
        'Tipo': 'Predicción'
    })

    print("="*80)

    return df_future, model_final, hist_metrics

In [18]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def create_interactive_plot_xgb(df_clean_full, wf_results, df_future, year_compare=2025):
    """
    Estilo referencia + fixes:
    - X hasta semana N
    - Y sin forzar a 0 (centrado/ajustado)
    - Conector entre último real y primera predicción
    - Bandas IC solo futuro
    """

    df_plot = df_clean_full.copy()
    df_plot["Fecha"] = pd.to_datetime(df_plot["Fecha"])
    df_plot["Año"] = df_plot["Fecha"].dt.year.astype(int)
    df_plot["Semana_ISO"] = df_plot["Fecha"].dt.isocalendar().week.astype(int)

    # --- Alinear predicciones WF
    train_end_first_fold = wf_results["fold_info"][0]["train_end"]
    start_idx_predictions = int(train_end_first_fold) + 1

    preds_wf = np.array(wf_results["predictions"], dtype=float) # Renamed to avoid clash with `preds` for `df_plot`
    n_preds = len(preds_wf)
    end_idx_predictions = min(start_idx_predictions + n_preds, len(df_plot))

    df_plot["Prediccion_WF"] = np.nan
    df_plot.loc[start_idx_predictions:end_idx_predictions-1, "Prediccion_WF"] = preds_wf[:(end_idx_predictions - start_idx_predictions)]

    # --- Filtrar años (no filtrados por semana aún)
    current_year = int(df_plot["Año"].max())
    df_current_raw = df_plot[df_plot["Año"] == current_year].copy()
    df_previous_raw = df_plot[df_plot["Año"] == year_compare].copy()

    # --- Futuro (no filtrado por semana aún)
    df_future_plot_raw = df_future.copy()
    df_future_plot_raw["Fecha"] = pd.to_datetime(df_future_plot_raw["Fecha"])
    df_future_plot_raw["Semana_ISO"] = df_future_plot_raw["Fecha"].dt.isocalendar().week.astype(int)
    df_future_plot_raw = df_future_plot_raw.sort_values("Semana_ISO").reset_index(drop=True)

    # Determine the dynamic max_week_display
    last_real_week = 0
    if not df_current_raw.empty:
        last_real_week = df_current_raw["Semana_ISO"].max()

    last_pred_week = 0
    if not df_future_plot_raw.empty:
        last_pred_week = df_future_plot_raw["Semana_ISO"].max()

    dynamic_max_week_display = max(last_real_week, last_pred_week)
    if dynamic_max_week_display == 0: # Fallback if no data at all
        dynamic_max_week_display = 10 # Default to 10 if no data available, or some other sensible default

    # Now filter using dynamic_max_week_display
    df_current = df_current_raw[df_current_raw["Semana_ISO"] <= dynamic_max_week_display].copy()
    df_previous = df_previous_raw[df_previous_raw["Semana_ISO"] <= dynamic_max_week_display].copy()
    df_future_plot = df_future_plot_raw[df_future_plot_raw["Semana_ISO"] <= dynamic_max_week_display].copy()

    # --- IC desde residuales WF (para el escalamiento)
    actuals = np.array(wf_results["actuals"], dtype=float)
    wf_preds = np.array(wf_results["predictions"], dtype=float)
    min_len = min(len(actuals), len(wf_preds))
    residuals = actuals[:min_len] - wf_preds[:min_len]
    std_base = float(np.std(residuals))

    # Calcular horizonte 'h' para escalamiento
    if not df_future_plot.empty:
        df_future_plot['h'] = np.arange(1, len(df_future_plot) + 1)
        scaled_std = std_base * np.sqrt(df_future_plot['h'])

        df_future_plot["IC_95_lower"] = df_future_plot["Precio_Predicho"] - 1.96 * scaled_std
        df_future_plot["IC_95_upper"] = df_future_plot["Precio_Predicho"] + 1.96 * scaled_std
        df_future_plot["IC_68_lower"] = df_future_plot["Precio_Predicho"] - 1.0 * scaled_std
        df_future_plot["IC_68_upper"] = df_future_plot["Precio_Predicho"] + 1.0 * scaled_std

    fig = go.Figure()

    # ========= IC 95% (detrás)
    if not df_future_plot.empty:
        fig.add_trace(go.Scatter(
            x=df_future_plot["Semana_ISO"],
            y=df_future_plot["IC_95_upper"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip"
        ))
        fig.add_trace(go.Scatter(
            x=df_future_plot["Semana_ISO"],
            y=df_future_plot["IC_95_lower"],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(255, 214, 170, 0.25)",
            name="Intervalo 95%",
            hoverinfo="skip"
        ))

        # ========= IC 68% (encima)
        fig.add_trace(go.Scatter(
            x=df_future_plot["Semana_ISO"],
            y=df_future_plot["IC_68_upper"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip"
        ))
        fig.add_trace(go.Scatter(
            x=df_future_plot["Semana_ISO"],
            y=df_future_plot["IC_68_lower"],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(255, 190, 120, 0.35)",
            name="Intervalo 68%",
            hoverinfo="skip"
        ))

    # ========= 2025 (gris punteado)
    if not df_previous.empty:
        fig.add_trace(go.Scatter(
            x=df_previous["Semana_ISO"],
            y=df_previous["Precio"],
            mode="lines+markers+text",
            name=str(year_compare) + " (Año Anterior)",
            line=dict(color="gray", width=2, dash="dot"),
            marker=dict(size=6, color="gray"),
            text=[f"${p:.2f}" for p in df_previous["Precio"]],
            textposition="top center",
            textfont=dict(color="gray", size=12),
            hovertemplate="<b>" + str(year_compare) + "</b>"
                         "<br>Semana: %{x}"
                         "<br>Precio: $%{y:.2f}"
                         "<extra></extra>"
        ))

    # ========= 2026 real (azul)
    if not df_current.empty:
        fig.add_trace(go.Scatter(
            x=df_current["Semana_ISO"],
            y=df_current["Precio"],
            mode="lines+markers+text",
            name=str(current_year) + " Real (YTD)",
            line=dict(color="steelblue", width=3),
            marker=dict(size=7, color="steelblue", symbol="circle"),
            text=[f"${p:.2f}" for p in df_current["Precio"]],
            textposition="bottom center",
            textfont=dict(color="steelblue", size=12),
            customdata=df_current["Fecha"].dt.strftime("%Y-%m-%d"),
            hovertemplate="<b>" + str(current_year) + " Real</b>"
                         "<br>Fecha: %{customdata}"
                         "<br>Semana: %{x}"
                         "<br>Precio: $%{y:.2f}"
                         "<extra></extra>"
        ))

    # ========= Conector: último real -> primera pred (para eliminar el “hueco”)
    if not df_current.empty and not df_future_plot.empty:
        # último punto real disponible
        last_real = df_current.dropna(subset=["Precio"]).sort_values("Semana_ISO").tail(1)
        first_pred = df_future_plot.dropna(subset=["Precio_Predicho"]).sort_values("Semana_ISO").head(1)

        if len(last_real) == 1 and len(first_pred) == 1:
            fig.add_trace(go.Scatter(
                x=[int(last_real["Semana_ISO"].iloc[0]), int(first_pred["Semana_ISO"].iloc[0])],
                y=[float(last_real["Precio"].iloc[0]), float(first_pred["Precio_Predicho"].iloc[0])],
                mode="lines",
                line=dict(color="steelblue", width=3),  # mismo grosor que real
                showlegend=False,
                hoverinfo="skip"
            ))

    # ========= Predicción futura (naranja)
    if not df_future_plot.empty:
        fig.add_trace(go.Scatter(
            x=df_future_plot["Semana_ISO"],
            y=df_future_plot["Precio_Predicho"],
            mode="lines+markers+text",
            name=str(current_year) + " Predicción",
            line=dict(color="darkorange", width=3, dash="dash"),
            marker=dict(size=8, color="darkorange", symbol="diamond"),
            text=[f"${p:.2f}" for p in df_future_plot["Precio_Predicho"]],
            textposition="top center",
            textfont=dict(color="darkorange", size=12),
            customdata=df_future_plot["Fecha"].dt.strftime("%Y-%m-%d"),
            hovertemplate="<b>" + str(current_year) + " Predicción</b>"
                         "<br>Fecha: %{customdata}"
                         "<br>Semana: %{x}"
                         "<br>Predicho: $%{y:.2f}"
                         "<extra></extra>"
        ))

    # ========= Eje Y “centrado”: rango basado en datos visibles (sin forzar 0)
    y_values = []
    for dfx, col in [(df_previous, "Precio"), (df_current, "Precio")]:
        if not dfx.empty and col in dfx.columns:
            y_values.extend(pd.to_numeric(dfx[col], errors="coerce").dropna().tolist())
    if not df_future_plot.empty:
        y_values.extend(pd.to_numeric(df_future_plot["Precio_Predicho"], errors="coerce").dropna().tolist())
        y_values.extend(pd.to_numeric(df_future_plot["IC_95_lower"], errors="coerce").dropna().tolist())
        y_values.extend(pd.to_numeric(df_future_plot["IC_95_upper"], errors="coerce").dropna().tolist())

    if len(y_values) > 0:
        y_min = min(y_values)
        y_max = max(y_values)
        pad = max(0.6, 0.08 * (y_max - y_min))  # padding agradable
        y_range = [y_min - pad, y_max + pad]
    else:
        y_range = None

    # semanas reales del forecast
    start_week = 1
    end_week = dynamic_max_week_display # Use the dynamic max week
    if not df_future_plot.empty:
        start_week = int(df_future_plot["Semana_ISO"].min())


    # ========= Layout
    fig.update_layout(
        template="plotly_white",
        width=1400,
        height=600,
        hovermode="x unified",
        title=dict(
            text="<b>Predicción Precio Banano " + str(current_year) + " (S" + str(start_week) + "-S" + str(end_week) + ")</b>"
                 "<br><span style='font-size: 0.9em'>Comparativo Dinámico vs " + str(year_compare) + " con Intervalos de Confianza</span>",
            x=0.02,
            xanchor="left"
        ),
        xaxis_title="Semana ISO",
        yaxis_title="Precio (USD/caja)",
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.98,
            xanchor="left",
            x=0.02,
            bgcolor="rgba(255,255,255,0.7)"
        ),
        margin=dict(l=70, r=30, t=90, b=60),
    )

    # X hasta la semana máxima de los datos presentes
    fig.update_xaxes(dtick=1, range=[0.5, float(dynamic_max_week_display) + 0.5])

    # Y centrado (sin ir a 0)
    if y_range is not None:
        fig.update_yaxes(range=y_range)

    return fig

## 4.4.0 Modelos Baseline (Benchmarks)

Antes de evaluar XGBoost, es fundamental comparar contra modelos simples para verificar que la complejidad del modelo se justifica. Si XGBoost no supera significativamente un baseline naive, el modelo complejo no aporta valor real.

**Baselines implementados:**
- **Naive**: precio = precio de la semana pasada
- **Seasonal Naive**: precio = precio de la misma semana del año anterior
- **Media Movil (4 semanas)**: promedio de las ultimas 4 semanas

In [19]:
print("\n" + "="*80)
print("📊 EVALUACION DE MODELOS BASELINE (BENCHMARKS)")
print("="*80)

# Preparar datos limpios (misma logica que walk-forward)
df_baseline = df_master.dropna(subset=feature_names_ml + ['Precio']).reset_index(drop=True)
initial_train_size = 170
n_samples = len(df_baseline)

print(f"Observaciones totales: {n_samples}")
print(f"Ventana de evaluacion: {initial_train_size} a {n_samples} ({n_samples - initial_train_size} predicciones)")

# Almacenar resultados de cada baseline
baseline_results = {}

# ================================================================
# BASELINE 1: NAIVE (precio = precio semana anterior)
# ================================================================
naive_preds = []
naive_actuals = []
for i in range(initial_train_size, n_samples):
    naive_preds.append(df_baseline['Precio'].iloc[i-1])
    naive_actuals.append(df_baseline['Precio'].iloc[i])

naive_metrics = evaluar_modelo(naive_actuals, naive_preds)
naive_da = directional_accuracy(naive_actuals, naive_preds)
baseline_results['Naive (lag-1)'] = {**naive_metrics, 'DA': naive_da}

print(f"\n🔹 Naive (precio = semana anterior):")
print(f"   RMSE: {naive_metrics['RMSE']:.4f} | MAE: {naive_metrics['MAE']:.4f} | MAPE: {naive_metrics['MAPE']:.2f}% | R2: {naive_metrics['R2']:.4f} | DA: {naive_da:.1f}%")

# ================================================================
# BASELINE 2: SEASONAL NAIVE (precio = misma semana año anterior)
# ================================================================
seasonal_preds = []
seasonal_actuals = []
for i in range(initial_train_size, n_samples):
    if i >= 52:
        seasonal_preds.append(df_baseline['Precio'].iloc[i-52])
        seasonal_actuals.append(df_baseline['Precio'].iloc[i])

if seasonal_actuals:
    seasonal_metrics = evaluar_modelo(seasonal_actuals, seasonal_preds)
    seasonal_da = directional_accuracy(seasonal_actuals, seasonal_preds)
    baseline_results['Seasonal Naive'] = {**seasonal_metrics, 'DA': seasonal_da}
    print(f"\n🔹 Seasonal Naive (misma semana del año anterior):")
    print(f"   RMSE: {seasonal_metrics['RMSE']:.4f} | MAE: {seasonal_metrics['MAE']:.4f} | MAPE: {seasonal_metrics['MAPE']:.2f}% | R2: {seasonal_metrics['R2']:.4f} | DA: {seasonal_da:.1f}%")

# ================================================================
# BASELINE 3: MEDIA MOVIL 4 SEMANAS
# ================================================================
ma_preds = []
ma_actuals = []
for i in range(initial_train_size, n_samples):
    if i >= 4:
        ma_preds.append(df_baseline['Precio'].iloc[i-4:i].mean())
        ma_actuals.append(df_baseline['Precio'].iloc[i])

ma_metrics = evaluar_modelo(ma_actuals, ma_preds)
ma_da = directional_accuracy(ma_actuals, ma_preds)
baseline_results['Media Movil (4sem)'] = {**ma_metrics, 'DA': ma_da}

print(f"\n🔹 Media Movil (4 semanas):")
print(f"   RMSE: {ma_metrics['RMSE']:.4f} | MAE: {ma_metrics['MAE']:.4f} | MAPE: {ma_metrics['MAPE']:.2f}% | R2: {ma_metrics['R2']:.4f} | DA: {ma_da:.1f}%")

# ================================================================
# TABLA RESUMEN COMPARATIVA
# ================================================================
print("\n" + "="*80)
print("📋 TABLA COMPARATIVA DE BASELINES")
print("="*80)

import pandas as pd
df_baselines = pd.DataFrame(baseline_results).T
df_baselines = df_baselines[['RMSE', 'MAE', 'MAPE', 'R2', 'DA']]
df_baselines.columns = ['RMSE', 'MAE', 'MAPE (%)', 'R2', 'Dir. Accuracy (%)']
print(df_baselines.round(4).to_string())

print("\n⚠️ XGBoost debe superar estos baselines para justificar su complejidad.")
print("   Si RMSE(XGBoost) >= RMSE(Naive), el modelo no aporta valor predictivo real.")
print("\n" + "="*80)


📊 EVALUACION DE MODELOS BASELINE (BENCHMARKS)
Observaciones totales: 218
Ventana de evaluacion: 170 a 218 (48 predicciones)

🔹 Naive (precio = semana anterior):
   RMSE: 1.3512 | MAE: 1.1171 | MAPE: 12.21% | R2: 0.4947 | DA: 53.2%

🔹 Seasonal Naive (misma semana del año anterior):
   RMSE: 3.0389 | MAE: 2.5442 | MAPE: 26.13% | R2: -1.5563 | DA: 55.3%

🔹 Media Movil (4 semanas):
   RMSE: 1.8552 | MAE: 1.5753 | MAPE: 17.81% | R2: 0.0473 | DA: 40.4%

📋 TABLA COMPARATIVA DE BASELINES
                      RMSE     MAE  MAPE (%)      R2  Dir. Accuracy (%)
Naive (lag-1)       1.3512  1.1171   12.2123  0.4947            53.1915
Seasonal Naive      3.0389  2.5442   26.1291 -1.5563            55.3191
Media Movil (4sem)  1.8552  1.5753   17.8053  0.0473            40.4255

⚠️ XGBoost debe superar estos baselines para justificar su complejidad.
   Si RMSE(XGBoost) >= RMSE(Naive), el modelo no aporta valor predictivo real.



In [20]:
print("\n" + "="*80)
print("🎯 EJECUTANDO ANALISIS COMPLETO CON XGBOOST")
print("="*80)

# 1. Walk-Forward Validation (test_size=4 para metricas significativas por fold)
print("\n🔄 Paso 1: Walk-Forward Validation...")
wf_results_xgb = walk_forward_validation_xgb(
    df=df_master,
    features=feature_names_ml,
    target='Precio',
    initial_train_size=150,
    test_size=4,  # Cambiado de 1 a 4 para R2 significativo por fold
    params=xgb_params
)

# Calcular directional accuracy
da_xgb = directional_accuracy(wf_results_xgb['actuals'], wf_results_xgb['predictions'])
print(f"\n📈 Directional Accuracy (XGBoost): {da_xgb:.1f}%")

# Comparar con baselines
print("\n" + "-"*60)
print("📋 COMPARACION XGBoost vs Baselines:")
print("-"*60)
xgb_test_metrics = wf_results_xgb['avg_test_metrics']
print(f"{'Modelo':<25} {'RMSE':>8} {'MAE':>8} {'MAPE(%)':>8}")
print("-"*55)
for name, metrics in baseline_results.items():
    print(f"{name:<25} {metrics['RMSE']:>8.4f} {metrics['MAE']:>8.4f} {metrics['MAPE']:>8.2f}")
print(f"{'XGBoost (all features)':<25} {xgb_test_metrics['RMSE']:>8.4f} {xgb_test_metrics['MAE']:>8.4f} {xgb_test_metrics['MAPE']:>8.2f}")
print("-"*55)

# 2. Prediccion de 4 semanas futuras
print("\n🔮 Paso 2: Prediccion de 4 semanas futuras...")
df_future_xgb, model_final_xgb, hist_metrics_xgb = predict_future_weeks_xgb(
    df=df_master,
    features=feature_names_ml,
    target='Precio',
    n_weeks=4,
    params=xgb_params
)

# 3. Crear visualizacion interactiva
print("\n📊 Paso 3: Creando visualizacion interactiva...")
fig_xgb = create_interactive_plot_xgb(
    df_clean_full=wf_results_xgb['df_clean'],
    wf_results=wf_results_xgb,
    df_future=df_future_xgb,
    year_compare=2025
)

# 4. Mostrar grafica
print("\n📈 Mostrando grafica interactiva...")
fig_xgb.show()

print("\n" + "="*80)
print("✅ ANALISIS XGBOOST COMPLETADO EXITOSAMENTE")
print("="*80)


🎯 EJECUTANDO ANALISIS COMPLETO CON XGBOOST

🔄 Paso 1: Walk-Forward Validation...
🚀 INICIANDO WALK-FORWARD VALIDATION CON XGBOOST
Total de observaciones: 218
Tamaño inicial de entrenamiento: 150
Tamaño de test por fold: 4
Número de folds: 17
Fold 1/17 | Train: 0-149 | Test: 150-153 | Test RMSE: 1.5861 | Test MAPE: 24.56%
Fold 10/17 | Train: 0-185 | Test: 186-189 | Test RMSE: 1.5576 | Test MAPE: 11.58%
Fold 17/17 | Train: 0-213 | Test: 214-217 | Test RMSE: 1.8912 | Test MAPE: 28.37%
📊 MÉTRICAS GENERALES WALK-FORWARD VALIDATION (XGBOOST)
RMSE: 1.5390
MAE:  1.2749
MAPE: 14.10%
R²:   0.4298

📈 PROMEDIO MÉTRICAS DE TRAIN
RMSE: 0.3527
MAE:  0.2788
MAPE: 4.10%
R²:   0.9818

📉 PROMEDIO MÉTRICAS DE TEST
RMSE: 1.4386
MAE:  1.2749
MAPE: 14.10%
R²:   -5.0923

📈 Directional Accuracy (XGBoost): 53.7%

------------------------------------------------------------
📋 COMPARACION XGBoost vs Baselines:
------------------------------------------------------------
Modelo                        RMSE      MAE


✅ ANALISIS XGBOOST COMPLETADO EXITOSAMENTE


In [21]:
from IPython.display import display, Markdown

# ================================================================
# TABLA DE MÉTRICAS GENERALES DE WALK-FORWARD VALIDATION
# ================================================================
print("\n" + "="*80)
print("📊 TABLA DE MÉTRICAS GENERALES (WALK-FORWARD VALIDATION)")
print("="*80)

metrics_df = pd.DataFrame([wf_results_xgb['overall_metrics']]).T
metrics_df.columns = ['Valor']
metrics_df.index.name = 'Métrica'
metrics_df_formatted = metrics_df.style.format("{:.4f}", subset=['Valor'])

display(metrics_df_formatted)
print("="*80)

# ================================================================
# TABLA DE PREDICCIONES FUTURAS DETALLADA
# ================================================================
print("\n" + "="*80)
print("🔮 TABLA DE PREDICCIONES FUTURAS DETALLADA")
print("""Incluye intervalos de confianza del 68% y 95% junto con comparación YoY""")
print("="*80)

# Recalcular std_residuals base de la validación walk-forward
residuals = wf_results_xgb['actuals'] - wf_results_xgb['predictions']
std_base = np.std(residuals)

# Preparar df_future_detailed con las columnas de IC
df_future_detailed = df_future_xgb.copy()
df_future_detailed['Fecha'] = pd.to_datetime(df_future_detailed['Fecha'])
df_future_detailed['Semana_ISO'] = df_future_detailed['Fecha'].dt.isocalendar().week.astype(int)

# Asignar el horizonte (h) para el escalamiento del error
df_future_detailed = df_future_detailed.sort_values('Fecha').reset_index(drop=True)
df_future_detailed['h'] = np.arange(1, len(df_future_detailed) + 1)

# Escalamiento del Error_Std (1σ) por horizonte
df_future_detailed['Error_Std (1σ)'] = std_base * np.sqrt(df_future_detailed['h'])

df_future_detailed['IC_68_Min'] = df_future_detailed['Precio_Predicho'] - 1.0 * df_future_detailed['Error_Std (1σ)']
df_future_detailed['IC_68_Max'] = df_future_detailed['Precio_Predicho'] + 1.0 * df_future_detailed['Error_Std (1σ)']
df_future_detailed['IC_95_Min'] = df_future_detailed['Precio_Predicho'] - 1.96 * df_future_detailed['Error_Std (1σ)']
df_future_detailed['IC_95_Max'] = df_future_detailed['Precio_Predicho'] + 1.96 * df_future_detailed['Error_Std (1σ)']

# Obtener los precios reales de 2025 para las semanas correspondientes
current_year = df_master['Fecha'].dt.year.max()
df_previous_year_data = df_master[df_master['Fecha'].dt.year == (current_year - 1)].copy()
df_previous_year_data['Semana_ISO'] = df_previous_year_data['Fecha'].dt.isocalendar().week.astype(int)

df_previous_prices = df_previous_year_data[['Semana_ISO', 'Precio']].rename(columns={'Precio': 'Real_2025'})

# Fusionar con df_future_detailed
df_future_detailed = df_future_detailed.merge(
    df_previous_prices,
    on='Semana_ISO',
    how='left'
)

# Calcular Diferencia_YoY y Diferencia_YoY_pct
df_future_detailed['Diferencia_YoY'] = df_future_detailed['Precio_Predicho'] - df_future_detailed['Real_2025']
df_future_detailed['Diferencia_YoY_pct'] = (df_future_detailed['Diferencia_YoY'] / df_future_detailed['Real_2025']) * 100

# Renombrar 'Precio_Predicho' a 'Prediccion_2026'
df_future_detailed.rename(columns={'Precio_Predicho': 'Prediccion_2026'}, inplace=True)

# Seleccionar y ordenar columnas para mejor visualización
df_future_detailed = df_future_detailed[[
    'Semana_ISO',
    'Prediccion_2026',
    'Error_Std (1σ)',
    'IC_68_Min',
    'IC_68_Max',
    'IC_95_Min',
    'IC_95_Max',
    'Real_2025',
    'Diferencia_YoY',
    'Diferencia_YoY_pct'
]]

display(df_future_detailed.style.format({
    'Prediccion_2026': "${:.2f}",
    'Error_Std (1σ)': "{:.2f}",
    'IC_68_Min': "${:.2f}",
    'IC_68_Max': "${:.2f}",
    'IC_95_Min': "${:.2f}",
    'IC_95_Max': "${:.2f}",
    'Real_2025': "${:.2f}",
    'Diferencia_YoY': "${:.2f}",
    'Diferencia_YoY_pct': "{:.2f}%"
}))
print("="*80)



📊 TABLA DE MÉTRICAS GENERALES (WALK-FORWARD VALIDATION)


,Valor
Métrica,
RMSE,1.5390
MAE,1.2749
MAPE,14.0950
R2,0.4298



🔮 TABLA DE PREDICCIONES FUTURAS DETALLADA
Incluye intervalos de confianza del 68% y 95% junto con comparación YoY


,Semana_ISO,Prediccion_2026,Error_Std (1σ),IC_68_Min,IC_68_Max,IC_95_Min,IC_95_Max,Real_2025,Diferencia_YoY,Diferencia_YoY_pct
0,12,$7.64,1.50,$6.13,$9.14,$4.69,$10.59,$7.48,$0.16,2.13%
1,13,$7.61,2.13,$5.48,$9.73,$3.43,$11.78,$6.15,$1.46,23.66%
2,14,$8.02,2.61,$5.41,$10.63,$2.91,$13.13,$6.23,$1.79,28.75%
3,15,$8.52,3.01,$5.51,$11.53,$2.62,$14.42,$7.55,$0.97,12.85%


### 4.4.1 PREDICCIÓN PARA TODO EL AÑO 2026

In [22]:
print("\n" + "="*80)
print("🎯 EJECUTANDO ANÁLISIS DE PREDICCIÓN PARA TODO EL AÑO 2026 (SOLO Predicciones Futuras)")
print("="*80)

# El modelo será entrenado con TODOS los datos disponibles en df_master (hasta la semana 7 de 2026)
# y luego predecirá las semanas futuras a partir de ese punto.

# Determinar el número de semanas a predecir para cubrir el resto de 2026
# La última fecha en df_master es 2026-02-15 (Semana 7). Queremos predecir las semanas 8 a 52.
# Esto son 52 - 7 = 45 semanas.
n_weeks_to_predict_remaining_2026 = 45

# 1. Predicción de las semanas restantes de 2026 (desde la semana 8)
print(f"\n🔮 Paso 1: Entrenando modelo con datos hasta Semana 7 de 2026 y prediciendo {n_weeks_to_predict_remaining_2026} semanas futuras...")
df_future_2026_combined, model_final_2026_combined, hist_metrics_2026_combined = predict_future_weeks_xgb(
    df=df_master, # Entrenar modelo con todos los datos hasta la última fecha disponible
    features=feature_names_ml, # Usar TODAS las features originales
    target='Precio',
    n_weeks=n_weeks_to_predict_remaining_2026, # Predecir las semanas restantes del año
    params=xgb_params # Usar los parámetros originales de XGBoost
)

# 2. Crear visualización interactiva para el año completo (solo con predicciones)
print("\n📊 Paso 2: Creando visualización interactiva para XGBoost (solo predicciones futuras)...")
fig_xgb_2026 = create_interactive_plot_xgb(
    df_clean_full=df_master, # Se usa df_master para tener el contexto completo de las fechas y el año anterior para la comparación
    wf_results=wf_results_xgb, # Usar los resultados de WF original para métricas de error para los IC
    df_future=df_future_2026_combined, # Pasar SOLAMENTE las predicciones futuras
    year_compare=2025
)

# 3. Mostrar gráfica
print("\n📈 Mostrando gráfica interactiva para XGBoost (solo predicciones futuras)...")
fig_xgb_2026.show()

# 4. Mostrar la tabla de predicciones detallada para todo el año (solo predicciones)
print("\n" + "="*80)
print("🔮 TABLA DE PREDICCIONES (SOLO FUTURAS) PARA TODO EL AÑO 2026")
print("Incluye intervalos de confianza del 68% y 95% junto con comparación YoY")
print("="*80)

# Recalcular std_residuals base de la validación walk-forward (del modelo original completo)
residuals = wf_results_xgb['actuals'] - wf_results_xgb['predictions']
std_base = np.std(residuals)

# df_full_year_predictions_table ahora solo contendrá las predicciones futuras
df_full_year_predictions_table = df_future_2026_combined.copy()

df_full_year_predictions_table['Fecha'] = pd.to_datetime(df_full_year_predictions_table['Fecha'])
df_full_year_predictions_table['Semana_ISO'] = df_full_year_predictions_table['Fecha'].dt.isocalendar().week.astype(int)

# Asignar el horizonte (h) para el escalamiento del error
df_full_year_predictions_table = df_full_year_predictions_table.sort_values('Fecha').reset_index(drop=True)
df_full_year_predictions_table['h'] = np.arange(1, n_weeks_to_predict_remaining_2026 + 1)

df_full_year_predictions_table['Error_Std (1σ)'] = std_base * np.sqrt(df_full_year_predictions_table['h'])

df_full_year_predictions_table['IC_68_Min'] = df_full_year_predictions_table['Precio_Predicho'] - 1.0 * df_full_year_predictions_table['Error_Std (1σ)']
df_full_year_predictions_table['IC_68_Max'] = df_full_year_predictions_table['Precio_Predicho'] + 1.0 * df_full_year_predictions_table['Error_Std (1σ)']
df_full_year_predictions_table['IC_95_Min'] = df_full_year_predictions_table['Precio_Predicho'] - 1.96 * df_full_year_predictions_table['Error_Std (1σ)']
df_full_year_predictions_table['IC_95_Max'] = df_full_year_predictions_table['Precio_Predicho'] + 1.96 * df_full_year_predictions_table['Error_Std (1σ)']

# Obtener los precios reales de 2025 para las semanas correspondientes
current_year_for_compare = df_full_year_predictions_table['Fecha'].dt.year.max()
df_previous_year_data_compare = df_master[df_master['Fecha'].dt.year == (current_year_for_compare - 1)].copy()
df_previous_year_data_compare['Semana_ISO'] = df_previous_year_data_compare['Fecha'].dt.isocalendar().week.astype(int)

df_previous_prices_compare = df_previous_year_data_compare[['Semana_ISO', 'Precio']].rename(columns={'Precio': 'Real_2025'})

# Fusionar con df_full_year_predictions_table
df_full_year_predictions_table = df_full_year_predictions_table.merge(
    df_previous_prices_compare,
    on='Semana_ISO',
    how='left'
)

# Calcular Diferencia_YoY y Diferencia_YoY_pct
df_full_year_predictions_table['Diferencia_YoY'] = df_full_year_predictions_table['Precio_Predicho'] - df_full_year_predictions_table['Real_2025']
df_full_year_predictions_table['Diferencia_YoY_pct'] = (df_full_year_predictions_table['Diferencia_YoY'] / df_full_year_predictions_table['Real_2025']) * 100

# Renombrar 'Precio_Predicho' a 'Prediccion_2026' y 'Tipo' a 'Prediccion'
df_full_year_predictions_table.rename(columns={'Precio_Predicho': 'Prediccion_2026'}, inplace=True)
df_full_year_predictions_table['Tipo'] = 'Predicción'

# Seleccionar y ordenar columnas para mejor visualización
df_full_year_predictions_table = df_full_year_predictions_table[[
    'Semana_ISO',
    'Tipo',
    'Prediccion_2026',
    'Error_Std (1σ)',
    'IC_68_Min',
    'IC_68_Max',
    'IC_95_Min',
    'IC_95_Max',
    'Real_2025',
    'Diferencia_YoY',
    'Diferencia_YoY_pct'
]]

display(df_full_year_predictions_table.style.format({
    'Prediccion_2026': "${:.2f}",
    'Error_Std (1σ)': "{:.2f}",
    'IC_68_Min': "${:.2f}",
    'IC_68_Max': "${:.2f}",
    'IC_95_Min': "${:.2f}",
    'IC_95_Max': "${:.2f}",
    'Real_2025': "${:.2f}",
    'Diferencia_YoY': "${:.2f}",
    'Diferencia_YoY_pct': "{:.2f}%",
    'h': "{:.0f}"
}))
print("="*80)

# Exportar la tabla a un archivo CSV
df_full_year_predictions_table.to_csv('predicciones_banano_2026_future_only_xgb.csv', index=False)
print("✅ Tabla 'predicciones_banano_2026_future_only_xgb.csv' exportada exitosamente.")

print("\n" + "="*80)
print("✅ PREDICCIÓN PARA TODO EL AÑO 2026 COMPLETADA EXITOSAMENTE")
print("="*80)



🎯 EJECUTANDO ANÁLISIS DE PREDICCIÓN PARA TODO EL AÑO 2026 (SOLO Predicciones Futuras)

🔮 Paso 1: Entrenando modelo con datos hasta Semana 7 de 2026 y prediciendo 45 semanas futuras...

🔮 PREDICCIÓN DE LAS PRÓXIMAS 45 SEMANAS CON XGBOOST
Entrenando modelo final con 218 observaciones...
✅ Modelo final entrenado
   RMSE: 0.3513
   MAPE: 4.03%
   R²:   0.9822

Última fecha en datos históricos: 2026-03-15
Semana 1 (2026-03-22): $7.64
Semana 2 (2026-03-29): $7.61
Semana 3 (2026-04-05): $8.02
Semana 4 (2026-04-12): $8.52
Semana 5 (2026-04-19): $8.20
Semana 6 (2026-04-26): $8.01
Semana 7 (2026-05-03): $8.54
Semana 8 (2026-05-10): $8.81
Semana 9 (2026-05-17): $8.17
Semana 10 (2026-05-24): $7.88
Semana 11 (2026-05-31): $8.23
Semana 12 (2026-06-07): $8.15
Semana 13 (2026-06-14): $8.79
Semana 14 (2026-06-21): $8.98
Semana 15 (2026-06-28): $8.65
Semana 16 (2026-07-05): $8.59
Semana 17 (2026-07-12): $8.95
Semana 18 (2026-07-19): $9.07
Semana 19 (2026-07-26): $8.95
Semana 20 (2026-08-02): $8.76
Sema


🔮 TABLA DE PREDICCIONES (SOLO FUTURAS) PARA TODO EL AÑO 2026
Incluye intervalos de confianza del 68% y 95% junto con comparación YoY


,Semana_ISO,Tipo,Prediccion_2026,Error_Std (1σ),IC_68_Min,IC_68_Max,IC_95_Min,IC_95_Max,Real_2025,Diferencia_YoY,Diferencia_YoY_pct
0,12,Predicción,$7.64,1.50,$6.13,$9.14,$4.69,$10.59,$nan,$nan,nan%
1,13,Predicción,$7.61,2.13,$5.48,$9.73,$3.43,$11.78,$nan,$nan,nan%
2,14,Predicción,$8.02,2.61,$5.41,$10.63,$2.91,$13.13,$nan,$nan,nan%
3,15,Predicción,$8.52,3.01,$5.51,$11.53,$2.62,$14.42,$nan,$nan,nan%
4,16,Predicción,$8.20,3.36,$4.84,$11.57,$1.61,$14.80,$nan,$nan,nan%
5,17,Predicción,$8.01,3.69,$4.32,$11.69,$0.78,$15.23,$nan,$nan,nan%
6,18,Predicción,$8.54,3.98,$4.56,$12.52,$0.73,$16.34,$nan,$nan,nan%
7,19,Predicción,$8.81,4.26,$4.56,$13.07,$0.47,$17.15,$nan,$nan,nan%
8,20,Predicción,$8.17,4.51,$3.66,$12.69,$-0.68,$17.02,$nan,$nan,nan%
9,21,Predicción,$7.88,4.76,$3.12,$12.64,$-1.45,$17.20,$nan,$nan,nan%


✅ Tabla 'predicciones_banano_2026_future_only_xgb.csv' exportada exitosamente.

✅ PREDICCIÓN PARA TODO EL AÑO 2026 COMPLETADA EXITOSAMENTE


## 4.5 Visualización de Resultados para XGBoost v3 (Top 25 Features)

In [23]:
# Paso 1: Ver importancia de features
feature_importance = pd.DataFrame({
    'Feature': feature_names_ml,
    'Importance': model_final_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("🎯 TOP 15 FEATURES MAS IMPORTANTES:")
print(feature_importance.head(15))

# Paso 2: Re-entrenar con top 15 features (reducido de 25 para mejor ratio obs/features)
top_n = 15
top_features = feature_importance.head(top_n)['Feature'].tolist()
print(f"\n📊 Usando {top_n} features (ratio ~{218//top_n} obs/feature)")

xgb_params_v3 = {
    'n_estimators': 150,
    'max_depth': 3,               # Reducido de 4 a 3 (menos overfitting)
    'learning_rate': 0.05,
    'min_child_weight': 20,       # Aumentado de 15 (fuerza hojas mas grandes)
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'gamma': 1.0,                 # Aumentado de 0.8
    'reg_alpha': 1.5,             # Aumentado de 1.0
    'reg_lambda': 8.0,            # Aumentado de 5.0 (regularizacion mas fuerte)
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

wf_results_v3 = walk_forward_validation_xgb(
    df=df_master,
    features=top_features,
    target='Precio',
    initial_train_size=170,
    test_size=4,  # Cambiado de 1 a 4
    params=xgb_params_v3
)

# Directional accuracy
da_v3 = directional_accuracy(wf_results_v3['actuals'], wf_results_v3['predictions'])
print(f"\n📈 Directional Accuracy (XGBoost V3): {da_v3:.1f}%")

# Comparacion final
print("\n" + "="*60)
print("📋 COMPARACION FINAL: Baselines vs XGBoost V1 vs V3")
print("="*60)
v3_metrics = wf_results_v3['avg_test_metrics']
xgb_v1_metrics = wf_results_xgb['avg_test_metrics']
print(f"{'Modelo':<30} {'RMSE':>8} {'MAE':>8} {'MAPE(%)':>8}")
print("-"*60)
for name, metrics in baseline_results.items():
    print(f"{name:<30} {metrics['RMSE']:>8.4f} {metrics['MAE']:>8.4f} {metrics['MAPE']:>8.2f}")
print(f"{'XGBoost V1 (all features)':<30} {xgb_v1_metrics['RMSE']:>8.4f} {xgb_v1_metrics['MAE']:>8.4f} {xgb_v1_metrics['MAPE']:>8.2f}")
print(f"{'XGBoost V3 (top 15)':<30} {v3_metrics['RMSE']:>8.4f} {v3_metrics['MAE']:>8.4f} {v3_metrics['MAPE']:>8.2f}")
print("="*60)

🎯 TOP 15 FEATURES MAS IMPORTANTES:
                    Feature  Importance
31    Precio_rolling_mean_4    0.260949
25              Precio_lag1    0.187288
33    Precio_rolling_mean_8    0.076686
43                tendencia    0.043306
23  Indice_Fletes_BDRY_lag4    0.025107
26              Precio_lag2    0.024905
36    Precio_rolling_std_12    0.021919
37             Precio_roc_4    0.021201
41                  mes_sin    0.020327
24       Petroleo_WTI_lag10    0.019886
20            Humedad_lag12    0.016268
38               semana_año    0.015469
39               semana_sin    0.015151
9          Temperatura_lag8    0.013698
40               semana_cos    0.013595

📊 Usando 15 features (ratio ~14 obs/feature)
🚀 INICIANDO WALK-FORWARD VALIDATION CON XGBOOST
Total de observaciones: 258
Tamaño inicial de entrenamiento: 170
Tamaño de test por fold: 4
Número de folds: 22
Fold 1/22 | Train: 0-169 | Test: 170-173 | Test RMSE: 0.8785 | Test MAPE: 11.81%
Fold 10/22 | Train: 0-205 | Test: 206-

In [24]:
print("\n" + "="*80)
print(f"🎯 EJECUTANDO ANALISIS DE PREDICCION CON XGBOOST V3 (TOP {top_n} FEATURES)")
print("="*80)

# 1. Prediccion de 4 semanas futuras usando el modelo v3
print(f"\n🔮 Paso 1: Prediccion de 4 semanas futuras con XGBoost V3 (top {top_n})...")
df_future_v3, model_final_v3, hist_metrics_v3 = predict_future_weeks_xgb(
    df=df_master,
    features=top_features,  # Usar las top features seleccionadas
    target='Precio',
    n_weeks=4,
    params=xgb_params_v3
)

# 2. Crear visualizacion interactiva para el modelo v3
print("\n📊 Paso 2: Creando visualizacion interactiva para XGBoost V3...")
fig_xgb_v3 = create_interactive_plot_xgb(
    df_clean_full=wf_results_v3['df_clean'],
    wf_results=wf_results_v3,
    df_future=df_future_v3,
    year_compare=2025
)

# 3. Mostrar grafica
print("\n📈 Mostrando grafica interactiva para XGBoost V3...")
fig_xgb_v3.show()

print("\n" + "="*80)
print("✅ ANALISIS XGBOOST V3 COMPLETADO EXITOSAMENTE")
print("="*80)


🎯 EJECUTANDO ANALISIS DE PREDICCION CON XGBOOST V3 (TOP 15 FEATURES)

🔮 Paso 1: Prediccion de 4 semanas futuras con XGBoost V3 (top 15)...

🔮 PREDICCIÓN DE LAS PRÓXIMAS 4 SEMANAS CON XGBOOST
Entrenando modelo final con 258 observaciones...
✅ Modelo final entrenado
   RMSE: 0.8399
   MAPE: 9.97%
   R²:   0.8987

Última fecha en datos históricos: 2026-03-15
Semana 1 (2026-03-22): $7.37
Semana 2 (2026-03-29): $7.65
Semana 3 (2026-04-05): $8.10
Semana 4 (2026-04-12): $8.24

📊 Paso 2: Creando visualizacion interactiva para XGBoost V3...

📈 Mostrando grafica interactiva para XGBoost V3...



✅ ANALISIS XGBOOST V3 COMPLETADO EXITOSAMENTE


---
# 5. FASE 2: Modelos Apropiados para el Problema

Los resultados de Fase 1 muestran que XGBoost V1 (47 features) no supera al baseline Naive, y V3 (15 features) apenas empata. El problema fundamental es que **258 observaciones son insuficientes para XGBoost**. En esta fase probamos modelos mas simples y un ensemble.

## 5.1 Ridge y Lasso Regression

Modelos lineales regularizados que funcionan mejor con datasets pequenos (~258 obs). Usan las mismas top 15 features de V3 y el mismo walk-forward validation para comparacion justa.

In [25]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler

print("="*80)
print("5.1 RIDGE / LASSO / ELASTICNET - WALK-FORWARD VALIDATION")
print("="*80)

def walk_forward_validation_linear(df, features, target='Precio',
                                    initial_train_size=170, test_size=4,
                                    model_class=Ridge, model_params=None):
    """Walk-forward validation para modelos lineales con escalado."""
    if model_params is None:
        model_params = {}

    df_clean = df.dropna(subset=features + [target]).reset_index(drop=True)
    n_samples = len(df_clean)
    n_folds = (n_samples - initial_train_size) // test_size

    predictions = []
    actuals = []
    train_metrics_list = []
    test_metrics_list = []

    for fold in range(n_folds):
        train_end = initial_train_size + (fold * test_size)
        test_start = train_end
        test_end = test_start + test_size
        if test_end > n_samples:
            break

        X_train = df_clean.loc[:train_end-1, features].values
        y_train = df_clean.loc[:train_end-1, target].values
        X_test = df_clean.loc[test_start:test_end-1, features].values
        y_test = df_clean.loc[test_start:test_end-1, target].values

        # Escalar features (importante para modelos lineales regularizados)
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = model_class(**model_params)
        model.fit(X_train_s, y_train)

        y_train_pred = model.predict(X_train_s)
        y_test_pred = model.predict(X_test_s)

        predictions.extend(y_test_pred)
        actuals.extend(y_test)
        train_metrics_list.append(evaluar_modelo(y_train, y_train_pred))
        test_metrics_list.append(evaluar_modelo(y_test, y_test_pred))

    predictions_arr = np.array(predictions)
    actuals_arr = np.array(actuals)
    overall = evaluar_modelo(actuals_arr, predictions_arr)
    da = directional_accuracy(actuals_arr, predictions_arr)

    avg_train = {k: np.mean([m[k] for m in train_metrics_list]) for k in ['RMSE','MAE','MAPE','R2']}
    avg_test = {k: np.mean([m[k] for m in test_metrics_list]) for k in ['RMSE','MAE','MAPE','R2']}

    return {
        'overall_metrics': overall,
        'avg_train_metrics': avg_train,
        'avg_test_metrics': avg_test,
        'predictions': predictions_arr,
        'actuals': actuals_arr,
        'da': da,
        'n_folds': n_folds
    }

# Probar multiples valores de alpha para cada modelo
alphas = [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
linear_results = {}

print(f"\nFeatures utilizadas: {len(top_features)} (mismas que XGBoost V3)")
print(f"Walk-forward: initial_train=170, test_size=4\n")

# Ridge Regression
print("-"*60)
print("RIDGE REGRESSION")
print("-"*60)
best_ridge = {'rmse': 999, 'alpha': None}
for alpha in alphas:
    res = walk_forward_validation_linear(
        df_master, top_features, model_class=Ridge,
        model_params={'alpha': alpha}
    )
    rmse = res['overall_metrics']['RMSE']
    mae = res['overall_metrics']['MAE']
    mape = res['overall_metrics']['MAPE']
    print(f"  alpha={alpha:6.2f} -> RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}%")
    if rmse < best_ridge['rmse']:
        best_ridge = {'rmse': rmse, 'alpha': alpha, 'results': res}

print(f"  >>> Mejor Ridge: alpha={best_ridge['alpha']}, RMSE={best_ridge['rmse']:.4f}")
linear_results['Ridge'] = best_ridge

# Lasso Regression
print("\n" + "-"*60)
print("LASSO REGRESSION")
print("-"*60)
best_lasso = {'rmse': 999, 'alpha': None}
for alpha in alphas:
    res = walk_forward_validation_linear(
        df_master, top_features, model_class=Lasso,
        model_params={'alpha': alpha, 'max_iter': 10000}
    )
    rmse = res['overall_metrics']['RMSE']
    mae = res['overall_metrics']['MAE']
    mape = res['overall_metrics']['MAPE']
    print(f"  alpha={alpha:6.2f} -> RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}%")
    if rmse < best_lasso['rmse']:
        best_lasso = {'rmse': rmse, 'alpha': alpha, 'results': res}

print(f"  >>> Mejor Lasso: alpha={best_lasso['alpha']}, RMSE={best_lasso['rmse']:.4f}")
linear_results['Lasso'] = best_lasso

# ElasticNet (combina L1 y L2)
print("\n" + "-"*60)
print("ELASTICNET REGRESSION")
print("-"*60)
best_enet = {'rmse': 999, 'alpha': None}
for alpha in alphas:
    for l1_ratio in [0.2, 0.5, 0.8]:
        res = walk_forward_validation_linear(
            df_master, top_features, model_class=ElasticNet,
            model_params={'alpha': alpha, 'l1_ratio': l1_ratio, 'max_iter': 10000}
        )
        rmse = res['overall_metrics']['RMSE']
        if rmse < best_enet['rmse']:
            best_enet = {'rmse': rmse, 'alpha': alpha, 'l1_ratio': l1_ratio, 'results': res}

print(f"  >>> Mejor ElasticNet: alpha={best_enet['alpha']}, l1_ratio={best_enet.get('l1_ratio')}, RMSE={best_enet['rmse']:.4f}")
linear_results['ElasticNet'] = best_enet

# Tabla comparativa
print("\n" + "="*80)
print("COMPARACION: Modelos Lineales vs Baselines vs XGBoost")
print("="*80)
print(f"{'Modelo':<30} {'RMSE':>8} {'MAE':>8} {'MAPE(%)':>8} {'DA(%)':>8}")
print("-"*70)

# Baselines
for name, metrics in baseline_results.items():
    print(f"{name:<30} {metrics['RMSE']:>8.4f} {metrics['MAE']:>8.4f} {metrics['MAPE']:>8.2f} {metrics['DA']:>8.1f}")

# XGBoost
xgb_overall = wf_results_xgb['overall_metrics']
print(f"{'XGBoost V1 (47 feat)':<30} {xgb_overall['RMSE']:>8.4f} {xgb_overall['MAE']:>8.4f} {xgb_overall['MAPE']:>8.2f} {da_xgb:>8.1f}")
v3_overall = wf_results_v3['overall_metrics']
print(f"{'XGBoost V3 (15 feat)':<30} {v3_overall['RMSE']:>8.4f} {v3_overall['MAE']:>8.4f} {v3_overall['MAPE']:>8.2f} {da_v3:>8.1f}")

# Modelos lineales
for name, best in linear_results.items():
    r = best['results']
    m = r['overall_metrics']
    print(f"{name + ' (best alpha)':<30} {m['RMSE']:>8.4f} {m['MAE']:>8.4f} {m['MAPE']:>8.2f} {r['da']:>8.1f}")

print("="*80)

5.1 RIDGE / LASSO / ELASTICNET - WALK-FORWARD VALIDATION

Features utilizadas: 15 (mismas que XGBoost V3)
Walk-forward: initial_train=170, test_size=4

------------------------------------------------------------
RIDGE REGRESSION
------------------------------------------------------------
  alpha=  0.01 -> RMSE: 1.2635 | MAE: 1.0371 | MAPE: 12.19%
  alpha=  0.10 -> RMSE: 1.2631 | MAE: 1.0369 | MAPE: 12.19%
  alpha=  1.00 -> RMSE: 1.2612 | MAE: 1.0377 | MAPE: 12.22%
  alpha= 10.00 -> RMSE: 1.2912 | MAE: 1.0738 | MAPE: 12.72%
  alpha= 50.00 -> RMSE: 1.3952 | MAE: 1.1444 | MAPE: 13.58%
  alpha=100.00 -> RMSE: 1.4557 | MAE: 1.1867 | MAPE: 13.99%
  >>> Mejor Ridge: alpha=1.0, RMSE=1.2612

------------------------------------------------------------
LASSO REGRESSION
------------------------------------------------------------
  alpha=  0.01 -> RMSE: 1.2574 | MAE: 1.0387 | MAPE: 12.19%
  alpha=  0.10 -> RMSE: 1.3016 | MAE: 1.0949 | MAPE: 12.66%
  alpha=  1.00 -> RMSE: 1.6990 | MAE: 1.3529 | 

## 5.2 Hyperparameter Tuning de XGBoost con Optuna

Optimizamos hiperparametros usando walk-forward RMSE como objetivo. Forzamos modelos muy simples (arboles poco profundos, alta regularizacion) para evitar overfitting en un dataset de ~258 obs.

In [26]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("="*80)
print("5.2 HYPERPARAMETER TUNING CON OPTUNA")
print("="*80)

# Preparar datos limpios una vez
df_optuna = df_master.dropna(subset=top_features + ['Precio']).reset_index(drop=True)
n_samples_optuna = len(df_optuna)
initial_train_optuna = 170
test_size_optuna = 4

print(f"Observaciones: {n_samples_optuna}")
print(f"Features: {len(top_features)} (top features de V3)")
print(f"Walk-forward: initial_train={initial_train_optuna}, test_size={test_size_optuna}")
print(f"Trials: 50\n")

def optuna_objective(trial):
    """Objetivo: minimizar walk-forward RMSE."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 2, 4),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15),
        'min_child_weight': trial.suggest_int('min_child_weight', 15, 80),
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
        'gamma': trial.suggest_float('gamma', 0.5, 3.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 3.0, 30.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': 0
    }

    predictions = []
    actuals = []
    n_folds = (n_samples_optuna - initial_train_optuna) // test_size_optuna

    for fold in range(n_folds):
        train_end = initial_train_optuna + (fold * test_size_optuna)
        test_start = train_end
        test_end = test_start + test_size_optuna
        if test_end > n_samples_optuna:
            break

        X_train = df_optuna.loc[:train_end-1, top_features]
        y_train = df_optuna.loc[:train_end-1, 'Precio']
        X_test = df_optuna.loc[test_start:test_end-1, top_features]
        y_test = df_optuna.loc[test_start:test_end-1, 'Precio']

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, verbose=False)
        y_pred = model.predict(X_test)

        predictions.extend(y_pred)
        actuals.extend(y_test)

    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    return rmse

# Ejecutar optimizacion
study = optuna.create_study(direction='minimize', study_name='xgboost_banana')
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)

print(f"\nMejor RMSE encontrado: {study.best_value:.4f}")
print(f"\nMejores hiperparametros:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Evaluar el mejor modelo con walk-forward completo
best_xgb_params = {**study.best_params, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}

print(f"\nEvaluando mejor modelo con walk-forward validation completo...")
wf_results_optuna = walk_forward_validation_xgb(
    df=df_master,
    features=top_features,
    target='Precio',
    initial_train_size=170,
    test_size=4,
    params=best_xgb_params
)

da_optuna = directional_accuracy(wf_results_optuna['actuals'], wf_results_optuna['predictions'])
optuna_overall = wf_results_optuna['overall_metrics']

print(f"\nXGBoost Optuna - Resultados finales:")
print(f"  RMSE: {optuna_overall['RMSE']:.4f}")
print(f"  MAE:  {optuna_overall['MAE']:.4f}")
print(f"  MAPE: {optuna_overall['MAPE']:.2f}%")
print(f"  R2:   {optuna_overall['R2']:.4f}")
print(f"  DA:   {da_optuna:.1f}%")

# Comparacion rapida
print(f"\n{'Modelo':<30} {'RMSE':>8}")
print("-"*40)
print(f"{'Naive (lag-1)':<30} {baseline_results['Naive (lag-1)']['RMSE']:>8.4f}")
print(f"{'XGBoost V3 (manual)':<30} {v3_overall['RMSE']:>8.4f}")
print(f"{'XGBoost Optuna':<30} {optuna_overall['RMSE']:>8.4f}")
print("="*80)

5.2 HYPERPARAMETER TUNING CON OPTUNA
Observaciones: 258
Features: 15 (top features de V3)
Walk-forward: initial_train=170, test_size=4
Trials: 50



Best trial: 43. Best value: 1.38911: 100%|██████████| 50/50 [00:53<00:00,  1.08s/it]



Mejor RMSE encontrado: 1.3891

Mejores hiperparametros:
  n_estimators: 69
  max_depth: 4
  learning_rate: 0.1371
  min_child_weight: 18
  subsample: 0.7043
  colsample_bytree: 0.7682
  gamma: 1.2631
  reg_alpha: 1.7369
  reg_lambda: 4.0973

Evaluando mejor modelo con walk-forward validation completo...
🚀 INICIANDO WALK-FORWARD VALIDATION CON XGBOOST
Total de observaciones: 258
Tamaño inicial de entrenamiento: 170
Tamaño de test por fold: 4
Número de folds: 22
Fold 1/22 | Train: 0-169 | Test: 170-173 | Test RMSE: 0.8663 | Test MAPE: 10.79%
Fold 10/22 | Train: 0-205 | Test: 206-209 | Test RMSE: 1.2010 | Test MAPE: 15.19%
Fold 20/22 | Train: 0-245 | Test: 246-249 | Test RMSE: 0.4820 | Test MAPE: 5.01%
Fold 22/22 | Train: 0-253 | Test: 254-257 | Test RMSE: 2.0143 | Test MAPE: 30.58%
📊 MÉTRICAS GENERALES WALK-FORWARD VALIDATION (XGBOOST)
RMSE: 1.3891
MAE:  1.1648
MAPE: 13.83%
R²:   0.5513

📈 PROMEDIO MÉTRICAS DE TRAIN
RMSE: 0.7417
MAE:  0.5686
MAPE: 8.82%
R²:   0.9158

📉 PROMEDIO MÉTRICAS

## 5.3 Ensemble Ponderado

Combina las predicciones de Naive, Ridge y XGBoost Optuna con pesos optimizados. La logica: el Naive es dificil de superar en 1-step, pero el ensemble puede capturar patrones que ningun modelo individual detecta.

In [27]:
from itertools import product as iter_product

print("="*80)
print("5.3 ENSEMBLE PONDERADO - OPTIMIZACION DE PESOS")
print("="*80)

# Walk-forward ensemble: en cada fold, combinar predicciones de Naive + Ridge + XGBoost
df_ens = df_master.dropna(subset=top_features + ['Precio']).reset_index(drop=True)
n_ens = len(df_ens)
initial_ens = 170
test_ens = 4
n_folds_ens = (n_ens - initial_ens) // test_ens

# Determinar el mejor modelo lineal
best_linear_name = min(linear_results, key=lambda k: linear_results[k]['rmse'])
best_linear_alpha = linear_results[best_linear_name]['alpha']
best_linear_class = {'Ridge': Ridge, 'Lasso': Lasso, 'ElasticNet': ElasticNet}[best_linear_name]
best_linear_params = {'alpha': best_linear_alpha}
if best_linear_name == 'ElasticNet':
    best_linear_params['l1_ratio'] = linear_results[best_linear_name].get('l1_ratio', 0.5)
    best_linear_params['max_iter'] = 10000
elif best_linear_name == 'Lasso':
    best_linear_params['max_iter'] = 10000

print(f"Mejor modelo lineal: {best_linear_name} (alpha={best_linear_alpha})")
print(f"Walk-forward: {n_folds_ens} folds\n")

# Recolectar predicciones individuales por fold
naive_preds_wf = []
linear_preds_wf = []
xgb_preds_wf = []
actuals_wf = []

for fold in range(n_folds_ens):
    train_end = initial_ens + (fold * test_ens)
    test_start = train_end
    test_end = test_start + test_ens
    if test_end > n_ens:
        break

    y_test = df_ens.loc[test_start:test_end-1, 'Precio'].values
    actuals_wf.extend(y_test)

    # Naive: ultimo precio conocido
    for i in range(test_start, test_end):
        naive_preds_wf.append(df_ens['Precio'].iloc[i-1])

    # Ridge/Lasso
    X_train = df_ens.loc[:train_end-1, top_features].values
    y_train = df_ens.loc[:train_end-1, 'Precio'].values
    X_test = df_ens.loc[test_start:test_end-1, top_features].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    lin_model = best_linear_class(**best_linear_params)
    lin_model.fit(X_train_s, y_train)
    linear_preds_wf.extend(lin_model.predict(X_test_s))

    # XGBoost Optuna
    xgb_model = xgb.XGBRegressor(**best_xgb_params)
    xgb_model.fit(
        df_ens.loc[:train_end-1, top_features],
        df_ens.loc[:train_end-1, 'Precio'],
        verbose=False
    )
    xgb_preds_wf.extend(xgb_model.predict(X_test))

naive_arr = np.array(naive_preds_wf)
linear_arr = np.array(linear_preds_wf)
xgb_arr = np.array(xgb_preds_wf)
actuals_arr = np.array(actuals_wf)

# Grid search de pesos (paso 0.05)
weights_range = np.arange(0, 1.05, 0.05)
best_ensemble = {'rmse': 999}

for w_naive, w_linear in iter_product(weights_range, weights_range):
    w_xgb = 1.0 - w_naive - w_linear
    if w_xgb < -0.01 or w_xgb > 1.01:
        continue
    w_xgb = max(0, w_xgb)

    ensemble_pred = w_naive * naive_arr + w_linear * linear_arr + w_xgb * xgb_arr
    rmse = np.sqrt(mean_squared_error(actuals_arr, ensemble_pred))

    if rmse < best_ensemble['rmse']:
        best_ensemble = {
            'rmse': rmse,
            'w_naive': round(w_naive, 2),
            'w_linear': round(w_linear, 2),
            'w_xgb': round(w_xgb, 2),
            'predictions': ensemble_pred
        }

# Metricas del mejor ensemble
ens_metrics = evaluar_modelo(actuals_arr, best_ensemble['predictions'])
ens_da = directional_accuracy(actuals_arr, best_ensemble['predictions'])

print("MEJOR ENSEMBLE ENCONTRADO:")
print(f"  Pesos: Naive={best_ensemble['w_naive']}, {best_linear_name}={best_ensemble['w_linear']}, XGBoost={best_ensemble['w_xgb']}")
print(f"  RMSE: {ens_metrics['RMSE']:.4f}")
print(f"  MAE:  {ens_metrics['MAE']:.4f}")
print(f"  MAPE: {ens_metrics['MAPE']:.2f}%")
print(f"  R2:   {ens_metrics['R2']:.4f}")
print(f"  DA:   {ens_da:.1f}%")

# Guardar resultados del ensemble
ensemble_results = {
    'metrics': ens_metrics,
    'da': ens_da,
    'weights': {
        'naive': best_ensemble['w_naive'],
        'linear': best_ensemble['w_linear'],
        'xgb': best_ensemble['w_xgb']
    },
    'predictions': best_ensemble['predictions'],
    'actuals': actuals_arr
}

print("\n" + "="*80)

5.3 ENSEMBLE PONDERADO - OPTIMIZACION DE PESOS
Mejor modelo lineal: Lasso (alpha=0.01)
Walk-forward: 22 folds

MEJOR ENSEMBLE ENCONTRADO:
  Pesos: Naive=0.25, Lasso=0.6, XGBoost=0.15
  RMSE: 1.2428
  MAE:  1.0443
  MAPE: 12.27%
  R2:   0.6408
  DA:   59.8%



## 5.4 Modelo 1-step vs Multi-step

Separamos la estrategia de prediccion:
- **1-step** (semana 1): Usa Precio_lag1/lag2, donde el Naive es fuerte. El modelo tiene acceso al precio mas reciente.
- **Multi-step** (semanas 2-4): Sin lags cortos de precio (que no estarian disponibles en la practica). Solo usa features exogenas y lags largos.

In [28]:
print("="*80)
print("5.4 MODELO 1-STEP vs MULTI-STEP")
print("="*80)

# Definir features para cada horizonte
# 1-step: puede usar lags cortos de precio (Precio_lag1, lag2, rolling_mean_4)
# Multi-step: solo features que estarian disponibles sin conocer precios recientes

features_1step = [f for f in top_features]  # Todas las top features (incluyen lags cortos)
features_multistep = [f for f in top_features
                       if not any(x in f for x in ['Precio_lag1', 'Precio_lag2', 'Precio_rolling_mean_4'])]

print(f"\nFeatures 1-step: {len(features_1step)} features")
print(f"Features multi-step: {len(features_multistep)} features")
print(f"  Eliminadas para multi-step: {set(features_1step) - set(features_multistep)}")

# Walk-forward para modelo multi-step
df_ms = df_master.dropna(subset=features_1step + ['Precio']).reset_index(drop=True)
n_ms = len(df_ms)
initial_ms = 170

# Evaluar 1-step (test_size=1, prediccion directa)
print("\n" + "-"*60)
print("MODELO 1-STEP (semana siguiente)")
print("-"*60)

preds_1step = []
actuals_1step = []

for i in range(initial_ms, n_ms):
    X_train = df_ms.loc[:i-1, features_1step]
    y_train = df_ms.loc[:i-1, 'Precio']
    X_test = df_ms.loc[i:i, features_1step]
    y_test = df_ms.loc[i, 'Precio']

    model_1s = xgb.XGBRegressor(**best_xgb_params)
    model_1s.fit(X_train, y_train, verbose=False)
    pred = model_1s.predict(X_test)[0]

    preds_1step.append(pred)
    actuals_1step.append(y_test)

metrics_1step = evaluar_modelo(actuals_1step, preds_1step)
da_1step = directional_accuracy(actuals_1step, preds_1step)

print(f"  RMSE: {metrics_1step['RMSE']:.4f}")
print(f"  MAE:  {metrics_1step['MAE']:.4f}")
print(f"  MAPE: {metrics_1step['MAPE']:.2f}%")
print(f"  R2:   {metrics_1step['R2']:.4f}")
print(f"  DA:   {da_1step:.1f}%")
print(f"  vs Naive RMSE: {baseline_results['Naive (lag-1)']['RMSE']:.4f}")

# Evaluar multi-step (sin lags cortos de precio)
print("\n" + "-"*60)
print("MODELO MULTI-STEP (sin lags cortos de precio)")
print("-"*60)

if len(features_multistep) >= 3:
    wf_multistep = walk_forward_validation_xgb(
        df=df_master,
        features=features_multistep,
        target='Precio',
        initial_train_size=170,
        test_size=4,
        params=best_xgb_params
    )

    da_multistep = directional_accuracy(wf_multistep['actuals'], wf_multistep['predictions'])
    ms_overall = wf_multistep['overall_metrics']

    print(f"\n  RMSE: {ms_overall['RMSE']:.4f}")
    print(f"  MAE:  {ms_overall['MAE']:.4f}")
    print(f"  MAPE: {ms_overall['MAPE']:.2f}%")
    print(f"  R2:   {ms_overall['R2']:.4f}")
    print(f"  DA:   {da_multistep:.1f}%")
else:
    print("  Insuficientes features para multi-step despues de eliminar lags cortos")
    ms_overall = None

# Resumen
print("\n" + "-"*60)
print("RESUMEN 1-step vs Multi-step:")
print("-"*60)
print(f"{'Horizonte':<25} {'RMSE':>8} {'MAE':>8} {'MAPE(%)':>8} {'DA(%)':>8}")
print("-"*65)
print(f"{'Naive (1-step)':<25} {baseline_results['Naive (lag-1)']['RMSE']:>8.4f} {baseline_results['Naive (lag-1)']['MAE']:>8.4f} {baseline_results['Naive (lag-1)']['MAPE']:>8.2f} {baseline_results['Naive (lag-1)']['DA']:>8.1f}")
print(f"{'XGBoost 1-step':<25} {metrics_1step['RMSE']:>8.4f} {metrics_1step['MAE']:>8.4f} {metrics_1step['MAPE']:>8.2f} {da_1step:>8.1f}")
if ms_overall:
    print(f"{'XGBoost multi-step':<25} {ms_overall['RMSE']:>8.4f} {ms_overall['MAE']:>8.4f} {ms_overall['MAPE']:>8.2f} {da_multistep:>8.1f}")

print("="*80)

5.4 MODELO 1-STEP vs MULTI-STEP

Features 1-step: 15 features
Features multi-step: 12 features
  Eliminadas para multi-step: {'Precio_lag2', 'Precio_rolling_mean_4', 'Precio_lag1'}

------------------------------------------------------------
MODELO 1-STEP (semana siguiente)
------------------------------------------------------------
  RMSE: 1.3039
  MAE:  1.1044
  MAPE: 13.08%
  R2:   0.6046
  DA:   54.0%
  vs Naive RMSE: 1.3512

------------------------------------------------------------
MODELO MULTI-STEP (sin lags cortos de precio)
------------------------------------------------------------
🚀 INICIANDO WALK-FORWARD VALIDATION CON XGBOOST
Total de observaciones: 258
Tamaño inicial de entrenamiento: 170
Tamaño de test por fold: 4
Número de folds: 22
Fold 1/22 | Train: 0-169 | Test: 170-173 | Test RMSE: 1.2823 | Test MAPE: 20.37%
Fold 10/22 | Train: 0-205 | Test: 206-209 | Test RMSE: 1.3499 | Test MAPE: 19.65%
Fold 20/22 | Train: 0-245 | Test: 246-249 | Test RMSE: 0.9100 | Test MAPE

## 5.5 Held-Out Test Set y Tabla Comparativa Final

Reservamos las ultimas 20 semanas como test intocable. Ningun modelo fue entrenado ni tuneado con estos datos. Este es el **unico** resultado que importa para evaluar la capacidad predictiva real.

In [29]:
print("="*80)
print("5.5 HELD-OUT TEST SET (ULTIMAS 20 SEMANAS)")
print("="*80)

# Preparar datos
df_holdout = df_master.dropna(subset=top_features + ['Precio']).reset_index(drop=True)
n_total = len(df_holdout)
holdout_size = 20
train_end_ho = n_total - holdout_size

print(f"Total observaciones limpias: {n_total}")
print(f"Train: 0 a {train_end_ho-1} ({train_end_ho} obs)")
print(f"Held-out test: {train_end_ho} a {n_total-1} ({holdout_size} obs)")

if 'Fecha' in df_holdout.columns:
    print(f"Periodo held-out: {df_holdout['Fecha'].iloc[train_end_ho]} a {df_holdout['Fecha'].iloc[-1]}")

X_train_ho = df_holdout.loc[:train_end_ho-1, top_features]
y_train_ho = df_holdout.loc[:train_end_ho-1, 'Precio']
X_test_ho = df_holdout.loc[train_end_ho:, top_features]
y_test_ho = df_holdout.loc[train_end_ho:, 'Precio'].values

holdout_results = {}

# 1. Naive
naive_ho = df_holdout.loc[train_end_ho-1:n_total-2, 'Precio'].values
holdout_results['Naive (lag-1)'] = {**evaluar_modelo(y_test_ho, naive_ho), 'DA': directional_accuracy(y_test_ho, naive_ho)}

# 2. Media Movil 4 semanas
ma_ho = []
for i in range(train_end_ho, n_total):
    ma_ho.append(df_holdout['Precio'].iloc[i-4:i].mean())
ma_ho = np.array(ma_ho)
holdout_results['Media Movil (4sem)'] = {**evaluar_modelo(y_test_ho, ma_ho), 'DA': directional_accuracy(y_test_ho, ma_ho)}

# 3. Ridge (mejor alpha)
scaler_ho = StandardScaler()
X_train_ho_s = scaler_ho.fit_transform(X_train_ho.values)
X_test_ho_s = scaler_ho.transform(X_test_ho.values)

ridge_ho = best_linear_class(**best_linear_params)
ridge_ho.fit(X_train_ho_s, y_train_ho)
ridge_pred_ho = ridge_ho.predict(X_test_ho_s)
holdout_results[f'{best_linear_name}'] = {**evaluar_modelo(y_test_ho, ridge_pred_ho), 'DA': directional_accuracy(y_test_ho, ridge_pred_ho)}

# 4. XGBoost V3 (params manuales)
xgb_v3_ho = xgb.XGBRegressor(**xgb_params_v3)
xgb_v3_ho.fit(X_train_ho, y_train_ho, verbose=False)
xgb_v3_pred_ho = xgb_v3_ho.predict(X_test_ho)
holdout_results['XGBoost V3 (manual)'] = {**evaluar_modelo(y_test_ho, xgb_v3_pred_ho), 'DA': directional_accuracy(y_test_ho, xgb_v3_pred_ho)}

# 5. XGBoost Optuna
xgb_opt_ho = xgb.XGBRegressor(**best_xgb_params)
xgb_opt_ho.fit(X_train_ho, y_train_ho, verbose=False)
xgb_opt_pred_ho = xgb_opt_ho.predict(X_test_ho)
holdout_results['XGBoost Optuna'] = {**evaluar_modelo(y_test_ho, xgb_opt_pred_ho), 'DA': directional_accuracy(y_test_ho, xgb_opt_pred_ho)}

# 6. Ensemble (pesos optimizados)
w = ensemble_results['weights']
ens_pred_ho = w['naive'] * naive_ho + w['linear'] * ridge_pred_ho + w['xgb'] * xgb_opt_pred_ho
holdout_results['Ensemble'] = {**evaluar_modelo(y_test_ho, ens_pred_ho), 'DA': directional_accuracy(y_test_ho, ens_pred_ho)}

# TABLA FINAL
print("\n" + "="*80)
print("TABLA COMPARATIVA FINAL - HELD-OUT TEST SET (20 semanas)")
print("="*80)
print(f"\n{'Modelo':<30} {'RMSE':>8} {'MAE':>8} {'MAPE(%)':>8} {'R2':>8} {'DA(%)':>8}")
print("-"*75)

for name, metrics in holdout_results.items():
    rmse = metrics['RMSE']
    mae = metrics['MAE']
    mape = metrics['MAPE']
    r2 = metrics['R2']
    da = metrics['DA']
    # Marcar el mejor
    print(f"{name:<30} {rmse:>8.4f} {mae:>8.4f} {mape:>8.2f} {r2:>8.4f} {da:>8.1f}")

# Identificar el mejor modelo
best_model_name = min(holdout_results, key=lambda k: holdout_results[k]['RMSE'])
best_rmse = holdout_results[best_model_name]['RMSE']
naive_rmse_ho = holdout_results['Naive (lag-1)']['RMSE']
improvement = (1 - best_rmse / naive_rmse_ho) * 100

print("-"*75)
print(f"\nMejor modelo: {best_model_name}")
print(f"RMSE: {best_rmse:.4f} (vs Naive: {naive_rmse_ho:.4f})")
if improvement > 0:
    print(f"Mejora sobre Naive: {improvement:.1f}%")
else:
    print(f"Peor que Naive por: {-improvement:.1f}%")

# Criterios de exito
print("\n" + "-"*40)
print("CRITERIOS DE EXITO:")
print("-"*40)
best_m = holdout_results[best_model_name]
print(f"  RMSE < 1.30 (vs Naive):  {'CUMPLE' if best_m['RMSE'] < 1.30 else 'NO CUMPLE'} ({best_m['RMSE']:.4f})")
print(f"  MAPE < 12%:              {'CUMPLE' if best_m['MAPE'] < 12 else 'NO CUMPLE'} ({best_m['MAPE']:.2f}%)")
print(f"  DA > 55%:                {'CUMPLE' if best_m['DA'] > 55 else 'NO CUMPLE'} ({best_m['DA']:.1f}%)")
print(f"  R2 > 0 (positivo):       {'CUMPLE' if best_m['R2'] > 0 else 'NO CUMPLE'} ({best_m['R2']:.4f})")

print("\n" + "="*80)
print("FASE 2 COMPLETADA")
print("="*80)

5.5 HELD-OUT TEST SET (ULTIMAS 20 SEMANAS)
Total observaciones limpias: 258
Train: 0 a 237 (238 obs)
Held-out test: 238 a 257 (20 obs)
Periodo held-out: 2025-11-02 00:00:00 a 2026-03-15 00:00:00

TABLA COMPARATIVA FINAL - HELD-OUT TEST SET (20 semanas)

Modelo                             RMSE      MAE  MAPE(%)       R2    DA(%)
---------------------------------------------------------------------------
Naive (lag-1)                    1.2753   1.0530    13.51   0.5636     47.4
Media Movil (4sem)               2.0244   1.7488    23.00  -0.0995     26.3
Lasso                            1.1232   0.9238    12.49   0.6616     63.2
XGBoost V3 (manual)              1.2749   1.0144    13.98   0.5639     73.7
XGBoost Optuna                   1.2794   1.0026    13.75   0.5608     52.6
Ensemble                         1.1433   0.9236    12.38   0.6493     63.2
---------------------------------------------------------------------------

Mejor modelo: Lasso
RMSE: 1.1232 (vs Naive: 1.2753)
Mejora so